In [ ]:
#@title Lesson 3 of 26: Linear Regression and Gradient Descent
from IPython.display import HTML, display
display(HTML('<div style="background:linear-gradient(118deg, #0d0726 0%, #2a0b5e 38%, #8f0f6b 72%, #ff6a1a 100%);border-radius:22px;padding:38px 36px 30px;color:#fff;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;box-shadow:0 12px 40px rgba(60,10,90,.35)"><div style="font-size:12px;letter-spacing:.34em;font-weight:700;color:#ffd1ea">BASIC PRINCIPLES OF ARTIFICIAL INTELLIGENCE  ·  LESSON 3 OF 26</div><div style="font-size:42px;font-weight:800;line-height:1.08;margin:14px 0 14px;text-shadow:0 2px 18px rgba(255,80,160,.45)">Linear Regression and Gradient Descent</div><div style="font-size:16px;line-height:1.55;max-width:680px;color:#f5ecff">How to fit a model to data reliably: features, squared loss, batch and stochastic gradient descent, choosing a step size, and reading loss curves.</div><div style="margin-top:22px"><a href="https://youtu.be/i6xu_Kn9omU" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#fff;color:#4a0d66">▶ Watch the lesson</a><a href="https://www.youtube.com/playlist?list=PLDyCOTShSfCk" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:rgba(255,255,255,.16);color:#fff;border:1px solid rgba(255,255,255,.45)">All lessons</a><a href="https://colab.research.google.com/github/neural-modules/ai-principles-notebooks/blob/main/03_linear_regression_and_gradient_descent.ipynb" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#ffb22e;color:#1a0d00">Open in Colab</a></div></div><div style="background:#141033;color:#ece8ff;border-radius:18px;padding:22px 26px;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;font-size:15px;line-height:1.5;border:1px solid #2c2466"><div style="font-size:12px;letter-spacing:.3em;font-weight:700;color:#a87bff;margin-bottom:6px">HOW THIS NOTEBOOK WORKS</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ff3d9a">1</span>Choose <b>File &gt; Save a copy in Drive</b> first, so your work is saved to your own account.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#a87bff">2</span>Run the set-up cell, then work down the page in order. Later exercises reuse earlier answers.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#2ad4ff">3</span>Replace each <code>...</code> with your answer and run the cell. The check runs straight away.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ffb22e">4</span>Stuck? Open the 💡 hint under the exercise. Some exercises have two.</div><div style="margin-top:14px;font-size:14px"><span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#11c48a;color:#03251a;padding:3px 10px;border-radius:999px;font-weight:700">✅ correct</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ff4d6d;color:#fff;padding:3px 10px;border-radius:999px;font-weight:700">❌ not yet, with a pointer</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ffb22e;color:#2a1700;padding:3px 10px;border-radius:999px;font-weight:700">⏳ still blank</span></div><div style="margin-top:16px;font-size:12px;color:#9d95c9">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</div></div>'))

In [ ]:
#@title Set-up: run this cell first
import base64, contextlib, html as _html, math, sys, zlib
import numpy as np

try:
    import torch
except ImportError:
    torch = None
try:
    import einops
except ImportError:
    get_ipython().run_line_magic("pip", "install -q einops")
    import einops


from IPython.display import HTML, display
import matplotlib.pyplot as plt
from cycler import cycler

PALETTE = {"pink": "#ff3d9a", "cyan": "#2ad4ff", "amber": "#ffb22e", "violet": "#a87bff",
           "mint": "#3dffb4", "orange": "#ff7a3d", "ink": "#ece8ff", "panel": "#141033", "night": "#0c0922"}
plt.rcParams.update({
    "figure.figsize": (6.8, 3.8), "figure.facecolor": "#0c0922", "axes.facecolor": "#141033",
    "savefig.facecolor": "#0c0922", "axes.edgecolor": "#3a3170", "axes.labelcolor": "#ece8ff",
    "axes.titlecolor": "#ffffff", "axes.titleweight": "bold", "axes.titlesize": 12,
    "text.color": "#ece8ff", "xtick.color": "#b9b2e6", "ytick.color": "#b9b2e6",
    "axes.grid": True, "grid.color": "#ffffff", "grid.alpha": 0.08, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False, "lines.linewidth": 2.4, "image.cmap": "plasma",
    "axes.prop_cycle": cycler(color=["#2ad4ff", "#ff3d9a", "#ffb22e", "#a87bff", "#3dffb4", "#ff7a3d"]),
})

_CARD = {"pass": ("linear-gradient(90deg,#0fae7a,#3dffb4)", "#03251a"),
         "wrong": ("linear-gradient(90deg,#e0245e,#ff6b6b)", "#ffffff"),
         "todo": ("linear-gradient(90deg,#ff9f1c,#ffd166)", "#2a1700")}

def _say(state: str, text: str):
    bg, fg = _CARD[state]
    try:
        display(HTML(f'<div style="background:{bg};color:{fg};padding:10px 16px;border-radius:12px;'
                     f'font-weight:600;font-family:sans-serif;margin:4px 0;max-width:860px">{text}</div>'))
    except Exception:
        print(text)


if sys.version_info < (3, 12):
    print("This notebook uses Python 3.12 syntax. Colab runs 3.12 or newer, so use Colab if a cell fails.")

class NotDone(Exception):
    """Raised when an answer is still the ... placeholder."""

_results: dict[str, str] = {}

def _ready(x):
    if x is ... or x is None:
        raise NotDone
    if isinstance(x, (tuple, list)):
        for v in x:
            _ready(v)
    return x

def _array(x):
    if torch is not None and isinstance(x, torch.Tensor):
        return x.detach().cpu().double().numpy()
    return np.asarray(x, dtype=float)

def close(answer, expected, tol: float = 1e-6, rel: float = 1e-6) -> bool:
    """True when answer matches expected to within tol (absolute) or rel (relative)."""
    _ready(answer)
    a, b = _array(answer), _array(expected)
    return a.shape == b.shape and bool(np.allclose(a, b, atol=tol, rtol=rel))

def same(answer, expected) -> bool:
    """Exact match, for counts, strings and choices."""
    _ready(answer)
    if isinstance(answer, str) and isinstance(expected, str):
        return answer.strip().lower() == expected.strip().lower()
    return answer == expected

@contextlib.contextmanager
def checking(label: str, praise: str = ""):
    try:
        yield
    except NotDone:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · replace the ... with your answer and run the cell again.")
    except NameError as e:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · run the earlier cells first ({_html.escape(str(e))}).")
    except AssertionError as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · {_html.escape(str(e)) or 'not quite yet. Open the hint above and try again.'}")
    except (TypeError, AttributeError) as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            _results[label] = "todo"
            _say("todo", f"⏳ <b>Exercise {label}</b> · your code still returns ... or None. Finish it and run the cell again.")
        else:
            _results[label] = "wrong"
            _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    except Exception as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    else:
        _results[label] = "pass"
        _say("pass", f"✅ <b>Exercise {label}</b> · correct." + (f" {_html.escape(praise)}" if praise else ""))

_CHECKS: dict = {}

def check(label: str):
    """Run the check for one exercise by hand, for example check("3.6")."""
    fn, praise = _CHECKS[label]
    with checking(label, praise):
        fn()

def _auto_check(result):
    import re
    m = re.match(r"\s*# Exercise (\d+\.\d+)\b", result.info.raw_cell or "")
    if not m or m[1] not in _CHECKS:
        return
    if result.error_in_exec is not None:
        _results[m[1]] = "wrong"
        _say("wrong", f"❌ <b>Exercise {m[1]}</b> · the cell raised an error (see above). Fix it and run the cell again.")
        return
    check(m[1])

_ip = get_ipython()
for _f in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_f, "__name__", "") == "_auto_check":
        _ip.events.unregister("post_run_cell", _f)
_ip.events.register("post_run_cell", _auto_check)

def progress():
    """One bar per part: how many of its exercises pass so far."""
    parts: dict[str, list[str]] = {}
    for label, state in _results.items():
        parts.setdefault(label.split(".")[0], []).append(state)
    if not parts:
        return _say("todo", "No exercises checked yet. Run an exercise cell to start.")
    rows = []
    for s, states in sorted(parts.items(), key=lambda kv: int(kv[0])):
        done, n = states.count("pass"), len(states)
        rows.append(f'<div style="margin:5px 0;font-family:sans-serif;color:#ece8ff"><span style="display:inline-block;width:64px">Part {s}</span>'
                    f'<span style="display:inline-block;width:320px;height:12px;background:#2a2357;border-radius:6px;vertical-align:middle">'
                    f'<span style="display:block;width:{320 * done // n}px;height:12px;border-radius:6px;'
                    f'background:linear-gradient(90deg,#ff3d9a,#ffb22e)"></span></span> &nbsp;{done} of {n} checked</div>')
    display(HTML('<div style="background:#141033;padding:14px 18px;border-radius:14px">' + "".join(rows) + "</div>"))

_say("pass", f"✅ Ready. Python {sys.version.split()[0]} · NumPy {np.__version__} · "
            f"PyTorch {torch.__version__ if torch is not None else 'missing'}")

# The answer checks. Each runs by itself after you run its exercise cell.
exec(zlib.decompress(base64.b64decode("""
eNqtfety40aS7n8/BYI/tkmbognwJsqjOaNut8eOM2t73b077aNQMEASErENEhwAFKVx+KXOI5wnO/llVhWqikV1e2IdbpHEpS5Z
WXnPrMWHSXQd7faDtKrS5+7tcDDpR/FgqP4k8mdy1/ti8av7ZDxIcGuEJ2f9aIyf48Elnvz74vuf/vPdW+dxbmmIx4fyjfq5GOKl
4eASf8b05hdfrLP7aFGUdd099qOn68UHeuz5mvruXX0R0X9V1hyqXXRflGnTpda3Wbrrdo/Rl9xVLZ099aIL+/dzrxd9+WWU9EwP
D1W6PtPDE/1UI9fN9aN187zPrrnXXt9p2r13OsZEhtaO84nG9tzDF3c4MY+HOndnive99/Rbq0P1SHO9ernT8xPxYUIQoZXJmrQf
1U22r4PgOdYEnFvp5jjs3fHF+7KKFlG+i6p095B1+W31vLwzSPf7bEcd1LcX8R3NhHqhMap14Itq8g4IDfocazPMcpdtyqZbpMus
6Ef8UbswsHF0GOX3URFdX8uTET2cAQN5yAWGLC20yHef0czq/J/ZYkvIuczWVVluCRabslyf9LMqd6u0yXb0r3t7q96Kvo7i4bB9
l6amR41G+tFt56bTjzqv8ecN/nzbuSO0wnDv2oE0VZrvuhjOocp4AIcan/sqX+GToXydDOk/Na6fN7mz5/S73U2P57vBfKWZuwDe
HuXlf2ZVWXeprUG9SffZbXwnt7Fk11FMs6OHinyXFg+DLH94TIt6wxiHV95Hf+FhfB0V2a4rQ+31Btv0qdv7LFShLo4GPxK3HasP
jI8+8Ki6aa8M7SSPQpw87iF/cywX63zfPbpLTGSKOj3i4TG9m+gfSfQV3Zvyz6+i0SDR7ayzegVEv+9HajNdE7YlerXGwIvNdZxd
mP2ECQ8/DzQX1wou3fsu+t2AzuHrBX3tEax4GTYuKAy1KHeP2RMGVpTXF0yON/k1f+6uL2NN/WpBAlrfep+usm5R4jl6pl0+og9L
DDNvsqopy6KmbbBdEkI0ebmru0+Em4k1brzS4PkukfuJIv3DwWxiPYP/aKPedxuaQEpT68Y0KZCoaNmL/hzh8n037Tl37rtLXCBg
zt2WrOl/R9iZ2fB4Xx0yDZL6Yc1ENOsy0Tvs17SPsbOybH0dYwWxeBojBDC0NutyO6D300PRLKrdQxePqx3Uj8pDQ3+qdVbxwtKu
VuyMaCX9CNJL1bEFD4LFrmykHXduuukir5tuNdhn1fbQMOS7E0U8uQF6hB8d7Mt9txdCopbrEJG/zUGlfsVn+zDNRRNuNQmXPNN9
mxdlHv9wt5KGg5CiI2Z/ynKIn3zxxeL9L78u3vznL//19h3N4jcillf2Wg0HMSFQMlTrdMnrlAymPSapV3osI0Gz4TDGw72+jyFE
ef1npd0eE2RzL5Z744Tv/U6je3vz5nt7fJssLZrNc/uKknBi3do6f8yqh3z3cPLIeBIeHA2PdlZUF+Xx5J3hMNEN78q8fn4ROglD
h+aAkb97c/O3tzJyGrjfKg03HhqCuM7Thy5WzOBl4UpFci/ARsBwmcv/mT6Hdy36KkSwwKGfp2brwxbUep3f33cLbHlsa/q8jkan
Lci83d6+5t7oRaIvp28wJG10NKtmE4Mqu28RMtuXq42iBmFZ9HMowmeIS9KRRzFzvu/sb3BCEhpdevCybBW3whXv8Gfs70+KWPd5
Y8DxgV5SAHntwuSPQMGWLD54csWnoAEQ0PunsHi26B0aqdtGaA3lCRq1z2bWT9Qa2rqtr2riH6/vXIAGJRB6Cyznwy19uRMJRL4r
qeKZf/icV0jjJlt9XMSLuKtGktLGqZqoTrcZzyOvCdIPJKfVNDMa8QRbmwTGaEnMlaQXDDZq0vpj5/R1cAH//VvwuL6wvn508gOi
Zue4yVcbEPD9oamjtMqi3WG7zKo6KnfEgOtVWmT/q/PF4s33b9/873e3r+JB/OqOINNtp9OPXv0MWaofNdmWhsiyZkRoEG3z3YH4
md3sIHq3T7d8d50/5KpPkb2JRPFomg09nj01BPC6pmFs6Gn6OnhlEFP3nYRAWWcktxdlU9P8fuvcp9u8eIZwraVgfAfRwme5b/It
SetV53dAgyRi7v2+PFQguk20oxZpjHX0MXuuT+HO/dxazdDeGvZ4evZ97u4O4hB1gg7qLK1opnnN3ZnX+/yzXpUEE3WPXz3Xr5oc
tTw67dTMl27HpmNsuai8l2kedivsIt2XNCeDyHcGI/iWbsxGhcRDBWIyr26KogWfvH+fF0W2HkQ/UUOVaSk6loeCsKAEdI9ZUXzD
He3LfNfIgNImylKCEi/Ehp6iLZDu6iMJM6eoMAqiQpM+ZDaYOttynRVgOTa81FMWvKznFOCkDX7JhgetZENoS0PlwVfZOl81RINW
NNbsaY/N2Dk3LI0VHdrc1S48KAezrAfVqNCEGZN5lqG+KY989Ui62UavRHnYrZ0lHHlLOKIlPAXu2APuivrNumq23RPjzuROGUcs
CxF02l4PlqI5Bp9ENFYCH92L9sWhZg1LXeqc70qblPpKP48h4IxY4EPjwl+N3ekvoafQ+5ZYU76n5WT82qd5xUBM12u1jOX6sGpc
ZB97kBoHITUJoWG+u8+qbLfKSPkiOj2jIXRn/WjKY5mZ7ugKhpCz+tT5VCuTIUQ/fPSj8XzObeX06Ia31bGsPoqStnuO3CWfeBMh
aebVrEVoNB7NIgWifCXaHMPHGt4geg+kV4TjuCEGJRo1aEuqNwI9Gdiq0yA2ZU/pdl9koh18PkrRwg7GPTySTPSuaHvHAAnjhKQ1
1aHZ4PHOZ/euLaBu74l1lT56MHiyHG5PPILgvqG9hzEQ8Vqma3yAn+MOy6HWqky9VZkG0WsWQq9lQX9vOz+fkgiSg36m1VhmDano
pCpS39kTiQr1VWRxofs0J/rcOdfuf5wSRGr3P0gxlRZB6u7LolD0hlTY9MpiJ6R+tsjF7IRmf8zSj870Z970Z5AqGHJ6lLRLlkW2
JZkTwCs/RormGqo3wJhI1MQ7hInMa8wIFAj0QAJYeRnESv3C4jHpKjrSExNdAsxr8aADox/b1YjEVhgFbYT6HwfMeJs1bLlrjYg0
8F2Q0AH/qK++NvvREizXabS5stBvA/1G2RY3txDq1RiY6r3rRz//8sObt+96fKspC5iZWCvmJqNjTrvgGSzamhzEPXmdd7q04IxP
jSz6k/oWq81GglqV6qY0+SFkgKzKm476JLxRgoy15JfekpMC3/mJ8InZ/WoD+Z30Lc3cnHaou6IACaLlZ6Gi2VRZpkSOdb7evWoY
FWj20cddeRx0vOVOTqRwAT/tzgXss2yT/f6nn759h8XGKhOFof95pV9rsldnq5KGtyyfOi809a3XlPwfc1PfGjEvJar7iYaeOjAZ
HyG34muHpdp/4s+j2I3bceqhnmEIUBeAnyJ3W6uSeOJ9wuL9qxPoJS9sFtoHDsc1u0Xm7QIAc2dLdHRflVsGxWO+zsqrSL/DW+U1
Dbl5Ydu0Pc9h9WDLTU8LCdYldDmfuPsSSzAczCcOGBIPDEkQDKMzchHxwQXP6vNFEiadhPhKGMDq6Hf0FuAWX9UhWTzxBLmEBTn2
xYlOReLkjmgSEGCcDKk5aouuDE7ndEbW2+REkZ6BZHDTJWYVrbXUdzDWN0BmWTQMPyzTAUzc6BkgXQ5l4d6cgRFTMvWSA4yxBwzI
anRVKMYbJRJj3XfZA8k3jxlzlWbDV+jekqgjXDRCx2olFEK6STGnpsgCgJsEAUcUoqny5YGlJsJIPccvwxJsR41NRGG10v0oo0ae
o+WzfAmw6vzhIasbxYaYRzOPZJs7k5cK9zXkpIH7/IFWh60gRTOoD8s96GeXLqOR6+50MGV2J709DZZp1f3u7c37xY83/w42401t
VRZldX37883f3r5///a2k0Lh79zBMreCOU58bub2Y04yY0P3McYVOKfT3p3pljSsRZMTzLv3nb8bpVAzHJJIIc2UTVpEvzktsCmx
dzWI73/vSGM8TdKLuj0bWSYeskAeBp4wQ0fzJEGOtcpMW4oU2R0JFfj1WqPSQ9YAS6ryQPKe1q7BYl10UroYJIFVqo0NzMFqIkak
SaZEiU4xKywwP6T7eiFmFTDKrnKqs3zSEcW5yfYYTQzooqsVCXEyJWZfhDl4oXOmbfFVUssaaW9HMCuaX8ldi8+3Y+fWiHW9joEP
LEF1dKN+KkL+xr2rH3Z28tRbHIjFP/KcIQsQ6qxom2REzgtxHGBxsgsaNugP3SIUE3xxF2KTPpKwRmtWHncEkP8+1M2WthbJUwTM
6LBndVTR3R3RnCNcgLPA0gQFcmBOvWDNrAsPIX2Df4rUhJ7ovrRqNIFV1Dp18QiPmIhptMzT+huSr3cXsJReMSba0qPYBJ/6/KAD
rpkHLojR49akd2C+MhmOSGiviKCQSllWW9bwBtG7pmSgloLdhAbRONqmHzNtiKJ7j5lYNzZZug9A4zIEjaZc1Pu0Iqzy9DdfIFAW
gt+GivnHVyJhj64kCGXKn2yo+5gBszHQfVmLRqraGomm2nl5GGKB5uX4jRuEaakoGN56kjX9obnvaH+yj8IC86UH5sugdDAPblsZ
xIL4mGXN+Iw5E+VxBaYcYlH6QGKxTPa4IC5w1ggf9wY7rHXB2HlNOKBoIjF2856ABfe+0TdulVQZj+bA0uQOMychYM6sGTfuXpwj
j0omyC/RBEV9Hs3VFKlRM0nlIOER/UUPwTNtEL4qI/H9oShsgcBZo7m3RnNN1kH1CJisk2eE/xDSIHuOfZvHIPo7BAxsTYd61MHO
lT2T1Q6CYGB3xMMgQiyz3XoBuwH85RxL8O6H//P23eL12x+/JZpKBNq9ZMIOGB/wBnMqIaJ4mn+KiLvWxrXkRZZveC1+dtu+QHTa
wQmDNwy8ZBcLjAOsRlx3WByzO++cb/dkojyhkz5Wz+mOe6ivOxcXHacromYdR0h44ptd0cS7pB5H2//3f3sdwmX1wLN6QBlpAB0I
deqJInuA+ywgd3RuDDi12FGA4adsAEkry+4LeL0obcRDX8EaKsRsG6mbnHCLyZtGyazWFspaJGowL3bccpe1yMKNEltE0LV9KIj1
EQGXtKs0hJxBt9SHBTGKdS2eOuJmRLJnPe2Wqsoju6SUpoNLtHyHrTiqFKTOGHNrabivfaInsv+XiGQxwUq3nkZpqwb0y9Lw7u6M
+IPxiRwd0hHiE0031ioT5D3AeEQsH/IESE6VP5md/kAaA0kdjxmx0Nbgp1UtgjFs71VGMgkLXOsSSmZzAvTRGSOEwOcYc0AG6wRY
2yMHXXnmTZZ8hB4ReqYBYBMLy9e6MYhov3KTVhtCONg4KmpQWV7AmNc+UvOMjbIUcZvo2ALoyLMcjNhyAG/QpH1ePIGqmasTCLIx
RGyjAWCFbQ6yMQVc3XaKEtZlMEENwBCmM4Y3BhObgGHW1ewWkWNGqMf8VZ/O7BNv9gmrnBPi39gXIoEZdUzs5bCUwn8UmGvYsMA2
MDO+foRABF5OdMSIwjhS7tqh3hdpUyuExVOdF1tNnFbNtZ5uOzlt++UGOcbLM5yPfXN6a2RnO/4kYLiCMxqODIKYA/WRB/WQ+2p0
xqQhK57482xHDwszz9rpcux1CdvCcDCckgBdbtk+AZnxSHSY1Lll+ShKHlvK2bmHh4gLlcfWtqnwiSTQLZwvj+kurzcBrDAWhirm
IbjYHtgbG2pW5K8qvh1d3YlKHH1NP+Wr8SPC/NgcSzMUGrey30KzdgAw8QAAfflmCd2CVCSShNUsWP29z++bTS3sSLNQBu2Ke+EH
U36Xt0VZsY3C0J0ADKbn15KNwH0EBn5Fwhz9w0xHeoouEaKNfpHwArirO/UmB33zF/Om9VbfgAp6HV8jTGBSMQoMexYW/cpj0Q9F
xP198ddffvj2TnNaBhmYqlKY1S58qPIQKTsucGOxZINQIp4p/bhuoRUXSCUoYBpqAw0+S0iUEZKAyFPwxDZt22llKRKpm6yivU4j
ujNTjlsqg6hI34K0z3cfIf2ZC8SqqFW6Ul9PiWb8kyMbr0dBKfAYEPxcZ0RI0nvLLIl3ay7OOglIUBDHbF8Q70aeAj5iBfwmInWI
sLxIBewQ3WBYBK5z45rOeE5VWSnt0NP08D4PmG1HZ9xXx0XdpFVfB9aBYnwpFONr5xrHQMN6eNiC0z0puem5L4TAXP4E/6T1OSGn
J74bzJaH5Wy8Sw9wUKmTSbTO4Q5Yw/QZE/8CbLKndNWQaJwIAdVuyGWWikiOPmiwQpNZg8MyKmim2GMkVKfMV+pvIkTH5RktgkRs
N3LdeErY7RkAd6vbL2gP0L/VQlRoDjM3MGV68aV7XYlg5tqv6tlTmKbLFZGztgeJXVDSiFmMvpbgjjBQ2wKLtYz60vMLS0jdwbP4
JfHikdYycS3W1+Qnm/2UEDIya2xQ/ERLt1lpDGujtepzb9WhpKe8FSa0GdSEUiJjxGSUXA2mtlf2V2xH0YJoJetMhMgJhnBUAUSs
8lgS2DpbET4p+yfMo6wuBSTzsK7epZ2wYE4C+s9Q6LHpVeQahsZXwiFo4VfpbpUV2vhKr3KIU161DL/KNO9wRemhL0sbJbGACxf7
sU6f2eaVgjHc09yJMjSsgxSWyLHNEZ1LgKJtwMSmxlYiSvMOQ2Cy0pR7gQeMvmIKkYHD3a+VTUC/3pal0hFyFZAFl+s+E0H+CNtp
CJTxCyy7rFYbn2Rsa4nDAL7qOCGt4dRi8aqb1ifjaSEnakhsG36EKrxn8CyrFLQePSmSa0v6WlwxcsPJ1MZn9DeArl7Q0lzw3tZa
nFFCumxkVptWz6LnTdSa1NhTrcZGtWrdV1qU1I4ekfvZzG1f5qFdRewuqtqpyfupJpLifCfa9ygDyitHGbw/VHBZBOARVtFAYhdH
2SRGubiIhypmRijD8CpAwNhngSF3TITw8ZFTSNqsROK5VmRvqOPjo63USKrZI4eL6c5/Oz7+fuXgmh5B96gT7p6cFUm8FYG6R1PS
dG44iG5afdnwGLTahnVG9yR6cWzQkR4hkhWAaFgR3B3WDxkwrKvVrGGMHCDzi7OA+CpBWC0nctx4V4Gjt9eOzrxG3rxGRri/mA/m
xJR5AIirwAKKXA8JEsTDhEXA/qZWj0camNf4jLgS64iqztBYNGO1LtTjOcEjqLefiB0SWHw89TONPa1uzFrddyCFQ8w1tjZYu3BM
KoHGUzwD7Tow0bBXeAnuiLSXNkVkZCGptrlLEu5sYkIlJM5F5QmdhkxYr3Re6HSkUj3wqKPw+/cYlITVukOENGFxcV96B7KPHEhO
PEhOAir5+Iwa16TLIlsczwBAR7J69zrn2sG2o6YuMMcLpBknnGwcD5IWoGZrar+yxo0zbWL5gxrbyYjv3Bhdr3kLYFMPYFPFr8zQ
NmnB9ka8j51wFYWmpDajlmkQ8fFM7CuFWh5/YwQeYuObKt19dFzQLEeBZTQZ6FN5TCuxMJPc0JTbAFrPzuzfMWZ/OWOGBzAENjGN
1Zn/zJs/VCZuYxC9xZQF+aQHsLTiXhOXKtuqHQ7qw3vwdKSX55zniwpuHhbE4cvpxhJo2rrP0aiSZpeZdMz+elY24DhypnHpTQMK
zFulqsCbLMZOAIGELyOqZk8s0RELYKGAZVYlqq7KrdrjgVmFfYvC8WgNjgu8h7jYrjC8sQ5TZY9SbHKuWWgNbYYxqPlXgImmfDrU
AdZs9ohDJ3NAMPdAAGk+1ITMjNugRRsJ5gqi1PnDTgvM1YEk2bQ4QtCt2clhtYG3SdQtQlKZkd0fygYJbiROrZ+7D+vFYdfkhUVz
4+yiJbuigH1EthNUueyp6UoAzce+bPGM3fZI8XYpdzwEv83vSTuqFbiPdOFP3HzvxKdCg+LITO6JpRCI4Jo/3eewgWkrDXvhzbLQ
ihx2a2OYK2gwJKdH3d+4qd81me6FAjqo0+GdkPh25DIEJIaJgMr5ShaQn8sD6wd7JiSuSDr0ZVKoKPHY8Apa2SUrGA5myfg5zXAQ
/bAjuZJ2CCcPpbTeO6WNpPcI1iVdPX+intV+oE1v+FCF/bJrGbK8lm/3VflIfYaQ4mzaFeuPCxaVEpaXkpbhsd8jiXXgTgCw+u1+
MOnzRR7w2ba2to/ORdk5sbaJVYx0/Pp6bF5MnzZQALutanXynrKutb7V4/VwcBk0pgk8/iWL2o0y+GIOVyQ51UxeGM1VHFBBpPEl
q9rYV+rGotTdgKERu4M0pppKmYtFTZoXwvmg7d0wNlmramG4cqAlQNcJWwdImoWWvQoEiU/OKHysBJPGN4ZkoPZ/KxlIJuKkF5Q5
RGpIdNyKbQaYeGrfhNU+w/qHbBSJSW8iLRzLwMZC6Pw6Ds1oI9R8lddYJSLZEx6ZcKPDLueIoGNgqmFdDsvIft0uK7WT1rrMLiFR
0bXqTNpnlZKWQUJK56WmPFfQSLmCtIaoUtFi8cleGd1MW1EmDtASD2jQzCbG88Z68SD6G/RhneOGGCcC4o7GUB5FFmI7bSrpG2ac
ASCF1bOsSRfb9IkTuyXgi3Uud5wjb5wj9iONB9FrOIdgYeFw1xyRuyvWuERSg0AGGY52b/2N8jD5D4e1yckZrcvEGFRdpluyqMoR
1wqUEBvEkYjMaolIJskUaMV8ImVs11a4UJINOqo5hHg2UfV/LoZKgB3yXxHMYyUMIPVWpEZ6hrHWRDVymKsNzrEHzlB61iScnsXJ
s0SxiFzVbdqs/VdlzfIHj5CLaMCkInzoIiZu34IRrN8Z3MQb3ETWGj7C1p6tXAwZKcxi2M7Y9C7j+kZkSJIWy3WQMoUVquNoYYJd
+mzQ7V4kSGDuAvDsNxwJhnKc0IU2geAuIf4yaCemRolVO7LECIHCWo6wyKsw8lFAgRw5AJp6AIIO9Bo2yzqFtVJrN0tYGrCXakR4
QnYkpgaz3jJdfWRFHdgE4B1BX3nXsb1QlANq5hRwYU2mu1rEI853p0+I0bGaqfhJDE0wRqOJ4kCJdp068tLE03ImrOW8Ma2AqNdl
a5XeHohNQoMzAlOiiFalqRbHvoL2swOv4vcC0wurPwXBiDZujQBVZUnrSQKbhcVwokDH0mZAHbagqQyBFbqQkg3syV56k73Uy4l+
geiJ+/Ig+g7+Fov2ti6bfqt5aStmzbq4QErZnJZyXdOqhrWpPLRRjOZ0rJWoDzyOEevOtGg0PBXZFfoumnLBWVcvKgfSLisExkzw
Z1ESmHKQEsJD1nuClRLLtsQKwD5luz295QB27gF2zrmbbcGaJq0aE8MLCmNMREqTj26g0tK8mionQQfutB0LC7RB6ozxyQBRUZuQ
GGQ0rO8VDJWXS7Hx24AjzLrEFXDs3xL+cdcLCwk1eHig1hT33Vr7jmVEt7Id3TwIqfk+6ZxlzmgzCdewknalVpXFvg0GVq1I4Ipr
Q19eG9qbnIeUqkCJkTiJpoPp5cs737ww1JuExyCOyWhXLsv1MxTq/aFREX5XzI6RL2IvJhNFxycXWlcj3x7THVRnwCAT1gskf//L
Dzd/W7x9f/OuxW7DBTKE8goTIAZomaLDa7B8XjAO9rmvNhBFb3E9cNGBeWuwDYBjZ5S5JG8E5bO1uxAngnPMDHekEoE52oWjIcT/
LtaciKWv97wviL/V7GlvEyWYL0tehziQFWM9heP0jJogW7Dm+kqEecotpLKmHyVtP7UEHciq1rymnj4wZX3gddYcM2TFDWYSmKLT
nJgiKjbM3mwUZmBfEWJXuPLGthT1fBuYwguuHKLNlv9F4tIvLklzRJGNgm1H2t3bOlUuiP2qr6FdaRpOTM1JmhGLoVPT7sgoNMr7
KMH/lofFAVfigSuUwzc9I8JDWyKxix0LUkCHx8S1f+C/15E5RpfjkNcCrmo995AATARvoZrWrqiTSENlcpSkFibdvG2DUxx5Uxxp
twUJcDPx0hiRSBx53IUVRXEhgouEs9uaqbjrT+F1znNTP6xh/HXti7GHKLZ0b9mECXmcaY29aY3FJHwp0bgqV5/ohs29VOEQZToU
G2FeZeEiBNNWE3BshKh2BNZrWXG0+u7ZCU+sa1ejO5XfO4i5mulkMr1rleNKBdid893IW2LEUCRfR2seFY8lUH7hlIW6HbMZILFq
a8mzF9fiommrP+FyuMCbi/Z6/nHrBjK9GAhIJamjsl92rOR/qUoH/oPQBmiLO2dhJ97CToJbcupxoaE/cb0m1rzxJE/bwj1c/MxJ
E2tpGRGbMt799dtWsBJ3OmRWTPCsb41bUf5LdhPwzlUODF7xoQOOqQcOqD3otwL9hwGFMSEezGkJpC0W7OJ+iz7Mm3jAalvUeHut
Ip9Ins+LgiPF1d2++Bi4soK+p3FSh1bQ0AM75pzLB0NhYcrb+YlVPXh06mPIWcucsvRzFK/CYQ8DFYx2DpRmHpRmHFqt4zhYUiiy
VCspFt+z8yxVGBDzkbos1l5sbmC+l2dtBXuilQs4IGAtyE/RkoUjCwgeBiIZVrLzSQlbZdr3kyBZUAcbADyauznAuPSAAdXqO8ZO
sSBOZHacoUeoouxeiDnivJAfka7S6gN9RnNk09XIYWXdYX+oGfcqFAvCggjyF/lHBlcdgtX8LDWtN4f7+yKzvQ5j+jILk1NjlKd2
4K8aEjpNkDsJcNEvKQrHSYzt5mTThQjQJAuGnR/9trCfNxDGzHWVHmE8Jil9E1n15WSfVdo5wx2gd5bPeDB/wIzPw/Bs8CpJmo3v
8SA5Md8z/pyx99uG+/iPme2PQVM9cEGSuGyj31X0j0Mu5hVS71iCVKYppCm9YLefeurqlNVVRZfEtWM6EkSUdsU9SN3RajSD6IdG
Ieim3O+BoAozk5DgbRTT6nw9QuWTOxKtbhrJZoRrLFh8dzoM1mP0C656tRjXphbjWtViBGX8WkLTqc+v4Ruy+NRaOKpHJ3pOo/Ta
VyQDhJItsdsWDFCJ9G3W3QC2T4dipJsMr+4MOaZntTedtaLJ0BQMMxkvWChSiDpnu14TlX02XWM2pg8QX9eErKEgQOirSPSI7SJY
XUdAcgN+7O5IxXQnrsrbwS4uYY2Z8d4T4nAenKtMDX1tSsdYbkqJkqCWEJzVuGEVYnk3SrvSsKCx16S8E1lg0FTAXw4fhysphKlB
fySKby0UW4bRcTEcDvFPYkvof32BgcsiwZXFyAGPvmVX+wa7SjKQ5NYJz3dBcqJghqq/zM6otw8LFFLTEQexFiE5HiZBgeawRgA8
8DOtWHvhIFrWDAJCh3TVahvdizEqSYf1C46pFF+UNd+Zp0/PxL82ZueECXMzutHF5CSOQVlGYXYez05ty7MzSjQNL1exWEoB1MxQ
FIgRhAOZj9J8tfNr/KJy+UKzWjLp+VqntgpEWhZhj/eJQvjp7tgHYGYxvusZxZnvcJzSeHBpFGfqL3tS/iQp9SgC6qs6pOnOPGV+
FlTmZ60yrzQHkKSyarqttdFV5m9Fnuspoqt/qdTT/QvVzVu1MOmFlAogKHdWt3pFyQHZu/zC1ejZdiAIjcE6xqyZp+LPWMVHoKng
c61UAdEUEJPJrmz9kxQ4FV2DTCuU6sq2CNCEBSJShlLW1EIi/6zV9euY7eQBQ4hw/wRsVMO4jltA/jE4BiJdy7zOrEgtMJh9otNX
6EcdS+JKy8cAydpOHLHuKduLA+CxB+Cxlc82jQfRz9zeSuWoCdNBswJjEkpp+ZA5EoDf5AyhlPh65QJVFdpipoJQiXirf8VflAOd
VnTojHnijVkcimwdgXLCqpQpC8oV0uSyjtsymmO6Bd8NczNjYw34UWYnDkfHUZKyHwXEuJ8Mh2LdHA9Pi3KYF5Z9lj8WS/a1TSGb
CzGfCCHiHSMm9OmYODRx9flUnGZpFE9JhJA9hTtWRbWxC7apBzbo2yMTF/KNqi82bVUL0xC2LrzaQ6NIB0ByxpXITGgBAURp8Qt4
tMSpSHBKWmFMypJVZclKBruahOwLH7t0JjPzJjMLUkSjxe6Wbt12paPJBls+L173AjXc7bXaLU0ZgHHf0HEuLIZw0XX2mIvOBHIm
qyFeeqsh9F8UHPCxrKnF2ysp2U7f4ivIyQl8ByhihLg4yd1Go1YcpuHMyTdcxVAI2G8Q9aEWnDQKwvJ7gH/RXUj9Wl62aNuHxesf
/kqMDB+YZ5UV1yRMGIlLBMH4NNNJ0QZ2coPOXGhLpab0n5lgWZQP9H/39Tsa3vJM0JepjCEFMYjw1gen5IbTiFSMxFz/Qdzw9Tsm
NRedF9RRaRbiuoeVr8NFNyzIvA4onIxmmgobBv/Jwhs/8msSbt+mrLoDak3kUvvsrD4684wnM4nRBZLJ+3qfKS8fbGRM47XjAls4
4jMHrqyBXPBAEKzKaXa8DAHiEDaRIPZooYgbk4Nx/4zWijjdkz15C1vJUrgsn8VCjd+JyXYMk62K3zemQXrCoqRjKS1V49mxToY+
KWbFIknXO+QI9pkBcS7weewROYaEeXmsIoolxExvgRIRA6AMltXEoS4vtH8eJI4aTh3bQ+polm3bc1oTroUWcw8t5mGtZ3jWIEgE
k9pcMDBhE0ykOBKDf9Iajy3QJ64iMvQ1EeiibFTl6onKmqtsadcSlCR+XHQcwXwFgy6SW8QEr4APWPehxFqRWadZKpdnNLouiv8U
2X2DAoTrBTtVOKRcn26kj0viRATQ7MAdZfx16tBqrqpyq+91He4mgBhExZZpwVZmLm9833TYvqpPajo3GHZCv/CQPMM1CTs8s057
HkKGnEdJ0q7s0w8uPZXxUlywiGxhjzEHcqvYQSPdcy4yC+GVSt7n3PK6Oaw+an8VAg2xjvk+sDhnIjE3tNKLh3SvawQ/XUVPOBZO
h9OpTJ5JW54aVOKpLWXEfsKYw6iuIm4O6iWUX64rRN9Damao34TPpeMSdyzSTlRUGPtvVWVsN0YV/lvkVwyt3FEH0IkH6JC+d2k5
bzlJGnT1j+JmODeEbtd9deyXwaJ00eM6Vc7F5YIz7/RPuUuP0pPLRQjzBdKYvLIqb/O12QgmkmW3dqPN9aAgHjkNKSkEXjsVG8H3
sIQOQEceQKFH/mhK8bTtMSK3aQabw3aP6tkcZCBBpqrKItcNh6lN3nNDiTb2OLjSKoo689FjAQwfB6vm1/qsshbRuk9s8VEVT7C0
cqgGKKIqwK7z0Mt9thN7omQz1ar7zgsdOTgz5kJQPRX2aYUZXdATamIAQd35rKGP1NmNcWjU7uZgWygnhkorV1LtC2CGH4Y1kbxx
Vnfsre44uF2C4a/NgfhzlzMLJYWBU9pPA1979gkpsuZNxqqRkO9TlJt4g4KWamqnskt8EEnBDWlO4T/SegQNlyoUWucnKyK1qjid
IlpVWW12zRG5zlxqqynLAI5Nz88dpWqsqeu1Y5Gk3oB0nKzmkw6dPb2FmCmmgk896740JpIREl3OANGqHeuAcuqBEporBifn7KBH
9MdUYECD49BdVRkOOj9vAfBO5kYCTnpd3MAMb65jNwzALVjhlEUaNbUWjxu/yg+fBXFSI8/aitYMZ94MOUBWdaCs8PSvJGRB5QOS
iMAyYFc4qeAhyUAcTs1MHHq3KljAB6AoyemYnXF1X54JmuVaiDzzNmlwrMPlFZUftSf8cG14nGMmp3aogzE0ecqtGrw2Cwz3KjRp
bJmLTjrKqvxR0s1bNmJRKuSasM997ED90oP6JWdLtANyEyeuDE2XsGVoujqVUUQct9Nlm7VXIdbsmD6rpBcWkmhCp7A3etIjuLlm
q/ZRlJL0yn/i4TAOqEU7xb+JqXZVmZvu423MdoE/RY+3VxekGkX/FjkXE7iseH+qQhn6EIN0J/UzTBHQ2gHh3AMhtIeEXl6lBVt7
t6nsNw7dYDsSr88WqyHCLaJyZUe0RzKJKxQ4uy0lMVQCpkMi4nzxQomN06w0O+PbPsTw1pyF15bmlws6e8v15AU8dOf6HU9e7rg9
CtB0bS6FOh9PTsNp5p5wPg+eXTBvxenVGhh2diCBWorQIQm1HklEXqvDBvEluXO0mrbKgpsuzDw7mBSzqP/BqePT9gwbKw4/P82a
nnvy8ZzlY1ZEdMwQSg+aoHJOCILht0UmjqqH30UywSQO/xRaRsJe1YtTcJnDKQPQWlflflFzoTvUYv1STmylZjTk6JskenVwX0ws
ykVyqJXZFVBkScGZ/MibfFulYjTS5ni7FI5GYzDeTco1tHIOclnqmSPZ7HTy43Pny0lGJh8Ox6fMnZzj/Lu2j+IZtnso8UjHEbDq
17l5/ebbzkn1EnEFt13cfoR/ik/itM5EpatSxERo7W8ff79yDsZRxtpGJ8paB/uxQ/Qprx2gjj2gsh8k+u8chduIlr5WZjgCGdvB
GIlSqfDXj95weSYBs5C7byVahskoX6hyZDQgAdTYJ85inRFUASrAoC9HBDjoN8ibbFt3e2H4AVw7VlhhP6IrgypjZ3yXsRbLJGeG
iVgP6csQm+jVb3jj91fWepkRmAX4A/3L4q289Wqj+BjZsyfCXAKSsyoTb1UgQ3+H0oowONaCXnDiqsx6FmIH0S+ZDmKXc07qOsfp
giJFk0qX7hxUgf1ZVR3ecqG+OrAm4Qwzjr5YYPx9PyFAJ0FzSkCAQqoiDgd1JBE9fqpJ6eb5kHfDn3RjsHcJNK06Dfo8VwuIUw+I
usQGLITCR8BR4GVDzMChUkV6OAGmFrMhnKouO1Rua0g4OnefYFqZ2nOnEAz7h4RO4qBNh1Dap/Qqiulc+iOk88RoCXjqPuVMXC68
VbbqlC6LJVjRePaZuSeuz6XQIZPguabAV6ZCDhuZiD8zYq4N+dF8sakOdaNLBTDZCgDvzPlcAPkC5yCwTP6YVl0dnt8G1CuVRBUJ
Mdel+Pe5No+hSobnysLOPUF6zoL0jQ7fnZvQ2rb+GMZ8wTKhYNNK70lVzo1uaHTjurG8DmxgvEcIpzpcTqJ/ak7uVhW3A7Cbn7Vc
g4qsFzRMmK2tTJ933//0y/s208cS5jKzp6M/h24MpfINn9DghE7VJ6k8EoL7QirP3BOu5yxc02aVcBh2kUMNOehM02Qoka+tB5eP
rVB4loov2dQsINJzehDccHGuEnpZGAcoVI+eWPfbwOo4WPOc34byu2hf5lelEbV5u3G/p6JhOGqsrYnNPhPpwz4+begfxDsMirt0
2ci7xwVHoBYLxwN1n9sneN7q6CxOZtCFmeHS0N/d2s0jE+Q9krLOiugH077UANqCz0abHasNVusSV8ZjJQfINijWr1NWSALB/usi
Hq7XCYYP8xx7MhAxdohistB1ULpzmKvp39wUCXYK7er4uTZ5Q1d2cxch8RchCS/C6FzwvJgTboNM886rRtJmT7dFvqUGtz2kkT+k
ESukgQIX4kOSI3U5aUPnHRCEN6HTwagxIxJ/qO0zQ7ocrDAK6OKM9bXkzXa79NZf2rfoebOlPtR8nLa5xwmzcle2C1qQxeyO+j13
zmN/zhBef4DBRQ48tuKKmVjgNIVnPteFPUB9Iaxo2yT9oZIFAVmRDkmpJLoRgknQrLo8PFg7PproPd9TTFpt/2VVpusVMWwOJJQn
Q1Ygak0sWTrKqptIaUkpAX571Y9+JAy+67XFfNuISk4uJMqXTPgAsTwLVbJh75TqxCv9zidGMvoxNDELF/4TH/5cFJwQjguOcKql
yJ7GLCjBU4N4atF/I8MJl/yxjLKqKqu+mE5SZUQSPSa0DmEJ1VS5FBru1aPHHvt1MkDgP03NEHYHuGKjV2dbPOtxmkhJtuFN7FMR
/qUxhHtVhxE9q7rE5tAaF/5TH/7TMBkyEuiHpapXLLNCPebVx+4th/XRVcKkmkvWBPjg0uaCgmlS9iXfWeRS2uVgZHNHTJWheN6l
qlbeZmMXtCP+0f2AwscLjmDFq9dAcYgYbl4d9DCRxvmtSApZ1eGOpEqu7tB2zo36w2GY2KoVr6QEn5tk5FHfmb8SrWAsGZXzIW+F
Jee1xXMRBjktyaQrmqgWlEOACbC1uCJzvD2skOaijryFTMi2nbbGUbrT9qcLE68cKmZNYzwjW+cN0giw5BLWK/mAKFaEQBU+X1Me
4UyDTyOTWkqvGZN/gzgNnXDTJtgI+z40HIjoJODxMVEYAoGyHevrH/5qdvMFiwgqlEt3PJLojGFQQFEtisCj5BO8JgKCrEr+YI7u
ELTmGsgKR8BkmMew0yNxkePSR45L9ojBNArUa0XJxMp0sqJGRolU0HCi00l70mNAYElofX353wQfPcGRr0rVIf/UVFIwEWt6XukD
iWXOjlJvR3++Zo+09gymy5qwoMks14cLhLkPBH0yF/rlbjj3B54alHLj/SFUkLbj2tTmN5S3Pmz70Vpiyll4ZsIK/2+Vhk8gH54r
781Z6Fm+K/dcGYUkQ3iaP4NMsujyGTtABBy72BUfba4IvqnAD35NtJN1qLUy4bG2zsW71X1PGxieqAPDMA84U4+bSKKqxu0T4c/e
2C6NDhFXqUD6L5LW+FTjcWp768QlIph8vPWPh+3PqO9d7gfRzaEpsbq6VMbaCURsMzvMaeLQsD8aBfYUh+JzMVMrLdOjyAV/hU9Z
RcSMvepBnHwPaoWA5uHA9s3Fsa/hxWENLz4TINTdLrg2UD/a8il6XKd+JtW23EwZq85WIER7WZZFt/wojfVw4of8Vifz2ZECvUBt
LNeZ8aqWpTelLGN3yok/5YQV/tlEF8KS4Uf1EeWLomPOwW66Mq6KgE9t/4sql9QAwpDi9XnI2pm0LOAaOQSXeHS2Bl6mYev6ucBf
Lnu2yikP8xk87qPyJMHr0kt5E2uoX+rsc+KG62ybF+XDszvCF2tGqjhfjLxzrh0efLAVfUCLaWXS+WN5qKwOdKkvIW+9zziKr47a
4/uM/VwtcloQ7XkhApgQbOQj2Mg9fg/i1yYr1pI1p6N8uE676yvz8ivEu/Epfxr1Nz5DNqZMMaYSgELdslFGskiZ3k5EGQNTOn/f
Iy8xO8ltChNP3f029sExduuSJeoc3NRqhQ3c8VQ2W9FWUvQKcmylMmEIBkZhfqJZP09VYoI3mb598Vd29g9Cp6RspzoRkNUnhs3T
1EsF1CHcsalFuVohZ0lV6OqrkVhlOy2LTCJ10tUjdyG5oTw2G7q9Wmnn5srSUgJeYVX/VImUdf50asmJfa06Zq3631vH8IVd28V2
9LZeXmG7MneaVFvLXRVPAN/kQwpZfYKOidR/amDLZWAeSj7LTLzigZwXGtO0XcvwOgZ5JG/3tTBJ+d4iPjUEwcvGdXVJJCQtCNn1
oqR00sSF39SH39RB7mEYt8dXkjymHRKKtrAlQ7LKai8MBkLZJASb2f8knh8XJAE/mLOcnnCAKjD2a/uKtv0cFVTtPLhJMJNWlbP6
zHb9A6LOnQ9lL8PMX4bZqdTW16kWg+FIpzQ69fxSfcAun2miStOt9QaquPgo1im0DGH11iz6Agy3q6uexiZQFwf/qlyTbdZUSkvX
JVDj4FmKuk3YdM+S6huVX3qDLXnzy9ubd+aktxYT96og2B8oMMFNWfS+/vTBbLrOROjw3TYZyDt9147EScIcHyPXZgyknarCzC+d
yhuo7vhpcUDqQZn221qVBggviwOXPm5eKtxUR+S44+9bR+ZVSgO9VxpC9M6Gikqh5NtyoLoo6zoGU5f71yIGh7SGj4mlQYWr/d8s
CtJVaFDbtNlIZtcI9f3B47yIxC4z8q8ic5AxtvfUKjZIsCOEFJHco6JzH0RzY9EaD+JLlYPJ4Af1TNxNIzGmBRscLOFgSRc+hrY4
GBBA8P8BlkXRlA==
""")).decode(), globals())


This notebook fits a line to data one small step at a time, and teaches you to tell when a run has gone wrong. It uses the same data as the video: five flats, each with a floor area and a sale price, and a model that predicts the price as $w$ times the area. The best weight is 2, the largest step size that stays stable is 0.4, and by the end you'll be able to work out that 0.4 from the data before running anything.

You'll work through the same ideas as the video, in code:

1. The four decisions
2. Features
3. The squared loss
4. Batch gradient descent
5. Choosing the step size
6. Stochastic gradient descent
7. Mini-batches
8. Convexity
9. Reading loss curves
10. The loop in NumPy and PyTorch
11. The payoff

Every section starts with a short explanation and a worked example you can run. Then come exercises that build on each other. Work them in order.

In [ ]:
#@title Helpers and imports
import itertools
import math
import time

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

rng = np.random.default_rng(0)
torch.manual_seed(0)


def draw_flats(ax, x, y, title=""):
    """Scatter the flats: area along the bottom, price up the side."""
    ax.scatter(x, y, s=70, color=PALETTE["cyan"], zorder=3)
    ax.set_xlabel("area (100 m²)")
    ax.set_ylabel("price (100 thousand)")
    if title:
        ax.set_title(title)


def draw_bowl(ax, ws, title="", x=None, y=None):
    """The training loss over w for the five flats, with the visited weights as dots."""
    grid = np.linspace(-0.5, 4.5, 300)
    ax.plot(grid, [_loss(g) for g in grid], color=PALETTE["violet"])
    ws = np.asarray(ws, dtype=float)
    ax.plot(ws, [_loss(w) for w in ws], "-o", color=PALETTE["amber"], ms=5, lw=1.2)
    ax.scatter([ws[0]], [_loss(ws[0])], color=PALETTE["pink"], s=60, zorder=4)
    ax.set_ylim(-0.5, 12)
    ax.set_xlabel("w")
    ax.set_title(title, fontsize=10)

Here are the two runs from the start of the video. Same five flats, same model, same method. Only the step size differs: 0.5 on the left and 0.05 on the right. The faint lines are the fitted line after each step, and the curve underneath is the loss.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.6, 6.2))
grid_x = np.linspace(0, 2.8, 2)
for col, (eta, steps, colour) in enumerate([(0.5, 8, PALETTE["pink"]), (0.05, 30, PALETTE["mint"])]):
    ws = _gd(0.0, eta, steps)
    ax = axes[0, col]
    for k, w in enumerate(ws):
        ax.plot(grid_x, w * grid_x, color=colour, alpha=0.15 + 0.85 * k / len(ws), lw=1.4)
    draw_flats(ax, _X5, _Y5, f"step size {eta}")
    ax.set_ylim(-8, 12)
    axes[1, col].plot([_loss(w) for w in ws], "-o", color=colour, ms=4)
    axes[1, col].set_xlabel("update"); axes[1, col].set_ylabel("loss")
plt.tight_layout()
plt.show()



## <font color="#ff2e88">The four decisions</font>

Suppose we want a program that guesses the price of a house. We collect examples, each one a house with its real sale price, and let the program find the pricing rule. These examples are the **training data**.

The program is a **predictor**: a function that takes an input and returns a guess. When the guess is a number, such as a price, the task is called **regression**.

Building a predictor from data always takes four decisions:

1. **The family of predictors.** The shape of the function, fixed before we look at data. Today it is the **linear** family: multiply each input number by a **weight** and add up the results. Every setting of the weights is one member of the family.
2. **The features.** The list of numbers that describes one input, called the **feature vector**.
3. **The loss.** One number that says how wrong a prediction is. Today it is the squared loss.
4. **The optimizer.** The procedure that searches the family for the weights with the lowest loss. Ours is gradient descent.

The first two decide what the predictor can express: that is **modeling**. The last two decide how its weights are found: that is **learning**. Once the weights are fixed, making a prediction is **inference**, one round of multiplying and adding.

The slots also tell you where to look when a model disappoints. If no line can follow the data, the family or the features are too weak. If a good line exists and the fit is still poor, the optimizer failed.

The cell below fills all four slots for six houses. `train_demo` is gradient descent, hidden away for now; you'll write it yourself in sections 4 and 10.

In [ ]:
HOUSES = [(120, 3), (80, 2), (95, 4), (150, 2), (60, 1), (110, 4)]   # (square metres, bedrooms)
PRICES = np.array([3.7, 2.5, 3.6, 4.0, 1.9, 3.8])                    # in hundreds of thousands


def features_v1(house):
    """Slot 2: the size in hundreds of square metres, and a constant 1."""
    size, bedrooms = house
    return np.array([size / 100, 1.0])


def predict_demo(w, phi):
    """Slot 1: the linear family. Multiply and add."""
    return float(w @ phi)


train_demo = _train          # slots 3 and 4: squared loss, gradient descent
w_v1, loss_v1 = train_demo(features_v1, HOUSES, PRICES)
print("weights", np.round(w_v1, 3), " training loss", round(loss_v1, 4))
print("prediction for a 100 m², 2-bedroom house:", round(predict_demo(w_v1, features_v1((100, 2))), 3))

fig, ax = plt.subplots()
sizes = np.array([h[0] for h in HOUSES]) / 100
ax.scatter(sizes, PRICES, color=PALETTE["cyan"], s=70, zorder=3)
grid_s = np.linspace(0.5, 1.6, 2)
ax.plot(grid_s, w_v1[0] * grid_s + w_v1[1], color=PALETTE["amber"])
ax.set_xlabel("size (100 m²)"); ax.set_ylabel("price (100 thousand)")
ax.set_title("Four decisions, one fitted line")
plt.show()



<font size="4"><b>Regression or not</b></font>

Five tasks, in order: the price of a house, whether an email is spam, tomorrow's temperature, which digit an image shows, the minutes a commute will take. Make a list of 5 booleans: `True` where the task is regression.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Regression means the output is a number on a scale. A choice from a list of labels is a different kind of task.

</details>

In [ ]:
# Exercise 1.1: running this cell checks your answer
is_regression = ...  # your answer



<font size="4"><b>Four slots for a commute</b></font>

A model predicts how long a commute will take. `COMMUTE_PARTS` lists its four pieces in a jumbled order. Make the dictionary `slots` that maps each slot name to the index of its piece.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Ask of each piece: is it the shape of the function, the numbers that go in, the score, or the search?

</details>

In [ ]:
# Exercise 1.2: running this cell checks your answer
COMMUTE_PARTS = ["gradient descent", "distance, hour of departure, raining or not",
                 "squared gap between predicted and real minutes", "weights times numbers, added up"]
slots = ...  # your answer



<font size="4"><b>Modeling or learning</b></font>

For each slot, say whether it belongs to modeling or to learning. Use the strings `"modeling"` and `"learning"`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Modeling decides what the predictor can express. Learning decides how its weights are found.

</details>

In [ ]:
# Exercise 1.3: running this cell checks your answer
stage = ...  # your answer



<font size="4"><b>Inference</b></font>

Write `predict(w, phi)`: the prediction of a linear predictor with weights `w` for the feature vector `phi`. Both are NumPy arrays. Return a plain float.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Multiply the arrays entry by entry and add the products. NumPy's `@` does both at once.

</details>

In [ ]:
# Exercise 1.4: running this cell checks your answer
def predict(w: np.ndarray, phi: np.ndarray) -> float:
    return ...  # your code here



<font size="4"><b>The cost of inference</b></font>

A prediction with `d` features takes `d` multiplications and `d - 1` additions. Write `inference_cost(d)` that returns the pair `(multiplications, additions)`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Adding up `d` products takes one fewer addition than there are products.

</details>

In [ ]:
# Exercise 1.5: running this cell checks your answer
def inference_cost(d: int) -> tuple[int, int]:
    return ...  # your code here



<font size="4"><b>One example&#x27;s loss</b></font>

Write `example_loss(w, phi, y)`: the squared gap between the prediction and the true value `y`. Use your `predict`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Take the prediction minus `y`, then square it.

</details>

In [ ]:
# Exercise 1.6: running this cell checks your answer
def example_loss(w: np.ndarray, phi: np.ndarray, y: float) -> float:
    return ...  # your code here



<font size="4"><b>Where to look</b></font>

Two runs disappoint. In run P, the training loss stops at 3.1, and a line with loss 0.2 is known to exist for the same features. In run Q, the loss stops at 3.1 whatever step size is tried, and the data curve upward so that no line can follow them. For each run, name the stage to blame: `"modeling"` or `"learning"`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

If a good line exists, the search missed it. If no good line exists, no search can find one.

</details>

In [ ]:
# Exercise 1.7: running this cell checks your answer
blame = ...  # your answer



<font size="4"><b>Swap one slot</b></font>

Keep the family, loss and optimizer, and change only the features. Write `features_v2(house)` that returns the size in hundreds of square metres, the number of bedrooms, and a constant 1, in that order. Then train with it: `w_v2, loss_v2 = train_demo(...)`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Copy `features_v1` and add one entry in the middle. Pass your new function to `train_demo` with `HOUSES` and `PRICES`.

</details>

In [ ]:
# Exercise 1.8: running this cell checks your answer
def features_v2(house):
    ...  # your code here


w_v2, loss_v2 = ..., ...  # your answer



## <font color="#7b2ff7">Features</font>

Take one house: 120 square metres, 3 bedrooms, neighbourhood B. A linear predictor can only multiply and add, so every field has to become a number. A **feature extractor** is a small function, written by us, that turns the raw input into the feature vector. It is written $\phi$ (the Greek letter phi).

* Fields that are already numbers go straight in. The size becomes 1.2, measured in hundreds of square metres, so it sits on a scale close to the other features. The bedrooms become 3. A plain number is right when "more" has a meaning.
* The neighbourhood is a **category**: a label from a fixed list, here A, B, C or D. Numbering the labels 1 to 4 would tell the predictor that D is 4 times A, and that the step from A to B equals the step from B to C. So each category gets a box of its own. The box for this house holds 1 and the others hold 0. That is a **one-hot vector**: exactly one entry is lit.
* One more feature is 1 for every house. Its weight is the **bias**, a base price added to every prediction.

The prediction is the **dot product** of the weights and the features: multiply each pair and add the products,
$$\text{prediction} = w \cdot \phi(x) = \sum_j w_j\, \phi_j(x).$$

A vector that is mostly zeros is **sparse**. A city with 500 neighbourhoods needs 500 boxes, and 499 of them hold 0 for any one house. Programs store only the non-zero entries, often as a dictionary from position to value.

In [ ]:
FEAT_NAMES = ["size", "bedrooms", "A", "B", "C", "D", "bias"]
W_HOUSE = np.array([2.0, 0.3, 0.0, 0.5, -0.2, 0.8, 0.4])
HOODS = ["A", "B", "C", "D"]
phi_house = np.array([1.2, 3, 0, 1, 0, 0, 1], dtype=float)

products = W_HOUSE * phi_house
print("features :", phi_house)
print("products :", np.round(products, 2))
print("prediction:", round(float(products.sum()), 2), " (4.2 means 420 thousand)")

fig, axes = plt.subplots(2, 1, figsize=(7.4, 3.6), sharex=True)
axes[0].imshow(phi_house[None, :], cmap="magma", vmin=-0.5, vmax=3.5, aspect="auto")
axes[1].imshow(products[None, :], cmap="cool", vmin=-0.5, vmax=2.5, aspect="auto")
for row, vals in enumerate([phi_house, products]):
    for j, v in enumerate(vals):
        axes[row].text(j, 0, f"{v:g}", ha="center", va="center", color=PALETTE["night"] if row == 1 or v > 1.5 else PALETTE["ink"], weight="bold")
    axes[row].set_yticks([0], ["features" if row == 0 else "w × features"]); axes[row].grid(False)
axes[1].set_xticks(range(7), FEAT_NAMES)
axes[0].set_title("One house as numbers, and what each number adds to the price")
plt.tight_layout()
plt.show()



<font size="4"><b>One box per category</b></font>

Write `one_hot(label, labels)` that returns a float NumPy array with one entry per label: 1 at the position of `label` and 0 everywhere else.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Start from `np.zeros(len(labels))` and set one entry. `labels.index(label)` finds the position.

</details>

In [ ]:
# Exercise 2.1: running this cell checks your answer
def one_hot(label: str, labels: list[str]) -> np.ndarray:
    ...  # your code here



<font size="4"><b>The feature extractor</b></font>

Write `featurize(size_m2, bedrooms, hood)` that returns the 7 features in the order of `FEAT_NAMES`: size in hundreds of square metres, bedrooms, the 4 neighbourhood boxes, and the bias feature. Use your `one_hot`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`np.concatenate` joins a short list, the one-hot array and `[1.0]` into one array.

</details>

In [ ]:
# Exercise 2.2: running this cell checks your answer
def featurize(size_m2: float, bedrooms: int, hood: str) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>The first price</b></font>

Use your `featurize` and `predict` to price the house of 120 square metres, 3 bedrooms, neighbourhood B, with `W_HOUSE`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

One line: build the features, then take the dot product with the weights.

</details>

In [ ]:
# Exercise 2.3: running this cell checks your answer
price_house = ...  # your answer



<font size="4"><b>Your turn: a flat in C</b></font>

Build the feature vector of a flat of 80 square metres with 2 bedrooms in neighbourhood C, and its predicted price with `W_HOUSE`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The size is 0.8 and the third neighbourhood box is lit.

</details>

In [ ]:
# Exercise 2.4: running this cell checks your answer
phi_try = ...  # your answer
price_try = ...  # your answer



<font size="4"><b>Read the products</b></font>

For the house in B, make `contributions`, the array of weight times feature, one entry per feature. Then store in `biggest` the name (from `FEAT_NAMES`) of the feature that adds the most. The bar chart appears when the check passes.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Multiply the two arrays entry by entry. `np.argmax` gives the position of the largest entry.

</details>

In [ ]:
# Exercise 2.5: running this cell checks your answer
contributions = ...  # your answer
biggest = ...  # your answer



<font size="4"><b>Why numbering fails</b></font>

Suppose the neighbourhoods were coded as one number, A = 1, B = 2, C = 3, D = 4, with a single weight of 0.3. Give the price steps from A to B and from B to C as `gaps_numbered`. Then give the same two steps under the one-hot weights of `W_HOUSE` as `gaps_onehot`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

With one number, each step of 1 in the code adds 0.3. With one-hot boxes, the step from A to B is the B weight minus the A weight.

</details>

In [ ]:
# Exercise 2.6: running this cell checks your answer
gaps_numbered = ...  # your answer
gaps_onehot = ...  # your answer



<font size="4"><b>A city with 500 neighbourhoods</b></font>

With 500 neighbourhoods, the feature vector holds the size, the bedrooms, 500 boxes and the bias. How many entries does it have, and how many of them are non-zero for one house (with at least 1 bedroom)? Give `(entries, non_zero)`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Of the 500 boxes, exactly one is lit.

</details>

In [ ]:
# Exercise 2.7: running this cell checks your answer
sizes_500 = ...  # your answer



<font size="4"><b>Store only what is lit</b></font>

Write `to_sparse(phi)` that returns a dictionary from position to value, holding only the non-zero entries of `phi`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Loop with `enumerate(phi)` and keep the pairs whose value is not 0. Convert each value with `float(...)`.

</details>

In [ ]:
# Exercise 2.8: running this cell checks your answer
def to_sparse(phi: np.ndarray) -> dict[int, float]:
    return ...  # your code here



<font size="4"><b>A sparse dot product</b></font>

Write `sparse_dot(w, sp)`: the dot product of a full weight array `w` with a sparse vector `sp` from `to_sparse`. Touch only the stored entries.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Add up `w[i] * v` over the items of the dictionary.

</details>

In [ ]:
# Exercise 2.9: running this cell checks your answer
def sparse_dot(w: np.ndarray, sp: dict[int, float]) -> float:
    return ...  # your code here



<font size="4"><b>A feature that bends</b></font>

Features can be built from other features. Take the features $[\text{size}, \text{size}^2, 1]$ with weights `W_BEND = [2.6, -0.3, 0.2]`. Make `bend_preds`, the prediction at every size in `SIZES_BEND`. The plot appears when the check passes.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Stack the three features as columns with `np.column_stack`, then multiply by the weights with `@`.

</details>

In [ ]:
# Exercise 2.10: running this cell checks your answer
SIZES_BEND = np.linspace(0.3, 3.0, 50)
W_BEND = np.array([2.6, -0.3, 0.2])
bend_preds = ...  # your answer



<font size="4"><b>Many houses at once</b></font>

Stack the features of three houses into a matrix `X_cards` with one row per house: (120, 3, B), (80, 2, C) and (95, 2, D). Then compute all three predictions with one matrix product.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`np.array([...])` of three feature vectors gives shape `(3, 7)`. Then `@ W_HOUSE`.

</details>

In [ ]:
# Exercise 2.11: running this cell checks your answer
X_cards = ...  # your answer
preds_cards = ...  # your answer



## <font color="#00a383">The squared loss</font>

Back to the five flats. So that we can draw every step, there is one feature, the floor area $x$ in hundreds of square metres, and one weight. The prediction is $w x$, a line through the origin whose slope is $w$. The price $y$ is in hundreds of thousands, so $w$ is the price of one unit of area.

Two flats have the same area and different prices, so no line passes through every dot. We need a score for every candidate line.

For each flat, the **residual** is the prediction minus the truth, $w x_i - y_i$. Square each residual, so every gap counts as positive and large gaps count far more: a gap twice as long gives a square 4 times as large. The average of the squares over the $n$ examples is the **training loss**:
$$L(w) = \frac{1}{n} \sum_{i=1}^{n} (w x_i - y_i)^2.$$

Everything in it comes from the data, apart from $w$. Plot $L$ for every $w$ and you get a parabola, a bowl with one lowest point. Learning means finding the bottom of this bowl.

In [ ]:
X5 = np.array([0.5, 1.0, 1.0, 2.0, 2.5])     # floor area, in units of 100 square metres
Y5 = np.array([1.2, 2.3, 1.7, 4.2, 4.8])     # price, in units of 100 thousand

fig, axes = plt.subplots(1, 2, figsize=(9.8, 4.2))
for ax, w, colour in [(axes[0], 1.0, PALETTE["pink"]), (axes[1], 2.0, PALETTE["mint"])]:
    ax.plot([0, 2.8], [0, 2.8 * w], color=colour)
    for xi, yi in zip(X5, Y5):
        r = w * xi - yi
        ax.add_patch(Rectangle((xi, min(yi, w * xi)), abs(r), abs(r), fc=colour, alpha=0.3, ec=colour))
        ax.plot([xi, xi], [yi, w * xi], color=colour, lw=1.2)
    draw_flats(ax, X5, Y5, f"w = {w:g}: training loss {_loss(w):.2f}")
    ax.set_aspect("equal"); ax.set_xlim(0, 5.2); ax.set_ylim(0, 5.6)
plt.tight_layout()
plt.show()



<font size="4"><b>Predictions and residuals</b></font>

For the line with $w = 1$, compute the 5 predictions `preds_w1` and the 5 residuals `resid_w1` (prediction minus truth).

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Both are one line of array arithmetic on `X5` and `Y5`.

</details>

In [ ]:
# Exercise 3.1: running this cell checks your answer
preds_w1 = ...  # your answer
resid_w1 = ...  # your answer



<font size="4"><b>Squares and their mean</b></font>

Square the residuals of $w = 1$ into `squares_w1`, and average them into `loss_w1`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`** 2`, then `np.mean`.

</details>

In [ ]:
# Exercise 3.2: running this cell checks your answer
squares_w1 = ...  # your answer
loss_w1 = ...  # your answer



<font size="4"><b>The training loss</b></font>

Write `train_loss(w, x, y)` for the model $w x$: the mean squared residual over arrays `x` and `y`. Return a plain float.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

One line: residuals, square, mean.

</details>

In [ ]:
# Exercise 3.3: running this cell checks your answer
def train_loss(w: float, x: np.ndarray, y: np.ndarray) -> float:
    return ...  # your code here



<font size="4"><b>A better line</b></font>

Compute the training loss of $w = 2$ with your function.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Call `train_loss` with the five flats.

</details>

In [ ]:
# Exercise 3.4: running this cell checks your answer
loss_w2 = ...  # your answer



<font size="4"><b>Where the loss comes from</b></font>

At $w = 1$, what share of the sum of squares comes from the two largest flats (the last two entries)?

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Add the last two entries of `squares_w1` and divide by the sum of all five.

</details>

In [ ]:
# Exercise 3.5: running this cell checks your answer
share_big = ...  # your answer



<font size="4"><b>Your turn: three points</b></font>

Three points $(1, 1)$, $(2, 6)$, $(3, 4)$ and the line with $w = 2$. Find its training loss.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Take each prediction minus the truth, square, then average.

</details>

In [ ]:
# Exercise 3.6: running this cell checks your answer
loss_three = ...  # your answer



<font size="4"><b>The bowl</b></font>

Compute the training loss at every weight in `W_GRID` as an array `bowl`, and the weight on the grid with the lowest loss as `w_grid_best`. The plot appears when the check passes.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

A list comprehension over `W_GRID` with your `train_loss`, then `np.argmin`.

</details>

In [ ]:
# Exercise 3.7: running this cell checks your answer
W_GRID = np.linspace(0, 4, 401)
bowl = ...  # your answer
w_grid_best = ...  # your answer



<font size="4"><b>The bottom by formula</b></font>

The slope of $L$ is zero at the bottom of the bowl. Setting $\frac{2}{n}\sum_i (w x_i - y_i) x_i = 0$ and solving gives $w^\star = \sum_i x_i y_i \,/\, \sum_i x_i^2$. Compute `w_star` and the lowest loss `loss_min`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Two sums with NumPy, then your `train_loss` at the result.

</details>

In [ ]:
# Exercise 3.8: running this cell checks your answer
w_star = ...  # your answer
loss_min = ...  # your answer



<font size="4"><b>The bowl&#x27;s coefficients</b></font>

Expand the square and $L(w) = a w^2 + b w + c$ with $a = \overline{x^2}$, $b = -2\,\overline{xy}$ and $c = \overline{y^2}$, where a bar means the mean over the flats. Compute `(a, b, c)`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Three `np.mean` calls. Check one value against your `train_loss` if unsure.

</details>

In [ ]:
# Exercise 3.9: running this cell checks your answer
abc = ...  # your answer



<font size="4"><b>Why square</b></font>

Two points $(1, 0)$ and $(1, 4)$, and the line with $w = 2$. Give the plain sum of the residuals and the squared loss, as `(sum_resid, sq_loss)`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

The residuals are $+2$ and $-2$.

</details>

In [ ]:
# Exercise 3.10: running this cell checks your answer
sum_resid, sq_loss = ..., ...  # your answer



<font size="4"><b>The loss in PyTorch</b></font>

PyTorch has the mean squared loss built in as `torch.nn.functional.mse_loss(prediction, target)`. Use it on tensors of the flats with $w = 2$, and store the result as a plain number.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`torch.tensor(X5)` makes a tensor. `.item()` turns a one-number tensor into a float.

</details>

In [ ]:
# Exercise 3.11: running this cell checks your answer
loss_torch = ...  # your answer



## <font color="#ff5f2e">Batch gradient descent</font>

Start with the line flat, $w = 0$. Every prediction is 0 and the loss is 10.06. To bring it down we move $w$, and the direction comes from the **gradient**: the slope of the loss with respect to $w$. Each example contributes 2 times its residual times its input, and the gradient of the training loss is the average:
$$\frac{dL}{dw} = \frac{2}{n} \sum_{i} (w x_i - y_i)\, x_i.$$

The **update rule** subtracts a small multiple of the gradient:
$$w \leftarrow w - \eta\, \frac{dL}{dw}.$$
The **step size** $\eta$ (eta) is a small positive number that we choose; here it is 0.1. The minus sign always sends the weight downhill: a negative gradient moves $w$ up, a positive one moves it down.

Each update reads all five flats. An update that uses every training example is a **batch** update, and the method is **batch gradient descent**. A plot of the loss after each update is a **loss curve**.

In [ ]:
w = 0.0
print(f"{'update':>6} {'w':>8} {'gradient':>9} {'loss':>8}")
for k in range(4):
    print(f"{k:>6} {w:>8.4f} {_grad(w):>9.4f} {_loss(w):>8.4f}")
    w = w - 0.1 * _grad(w)

fig, ax = plt.subplots()
for k, w in enumerate(_gd(0.0, 0.1, 3)):
    ax.plot([0, 2.8], [0, 2.8 * w], color=plt.cm.plasma(0.2 + 0.25 * k), label=f"after {k} updates: w = {w:g}")
draw_flats(ax, X5, Y5, "Batch gradient descent tilts the line up, step size 0.1")
ax.legend(fontsize=8)
plt.show()



<font size="4"><b>The cells at w = 0</b></font>

At $w = 0$, compute the array `cells_0`, holding residual times area for each flat.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Residuals first, then multiply entry by entry by `X5`.

</details>

In [ ]:
# Exercise 4.1: running this cell checks your answer
cells_0 = ...  # your answer



<font size="4"><b>The gradient</b></font>

Write `grad_w(w, x, y)`: the gradient of the training loss for the model $w x$. Return a plain float.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Two times the mean of residual times input.

</details>

In [ ]:
# Exercise 4.2: running this cell checks your answer
def grad_w(w: float, x: np.ndarray, y: np.ndarray) -> float:
    return ...  # your code here



<font size="4"><b>Check it with a nudge</b></font>

Move $w$ from 0 to 0.01 and divide the change in the training loss by 0.01. Use your `train_loss`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The loss at 0.01 minus the loss at 0, over 0.01.

</details>

In [ ]:
# Exercise 4.3: running this cell checks your answer
nudge_0 = ...  # your answer



<font size="4"><b>The first update</b></font>

Take one update from $w = 0$ with step size 0.1, and compute the new loss.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The new weight is the old one minus the step size times your `grad_w`.

</details>

In [ ]:
# Exercise 4.4: running this cell checks your answer
w_1 = ...  # your answer
loss_1 = ...  # your answer



<font size="4"><b>Batch gradient descent</b></font>

Write `batch_gd(w0, eta, steps, x, y)` that returns a NumPy array of every weight visited, the start included, so it has `steps + 1` entries. Use your `grad_w`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Keep a list that starts as `[w0]` and append one updated weight per step.

</details>

In [ ]:
# Exercise 4.5: running this cell checks your answer
def batch_gd(w0: float, eta: float, steps: int, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    ...  # your code here



<font size="4"><b>The table</b></font>

Run 3 updates from 0 with step size 0.1. Make three arrays, one entry per weight visited (4 entries each): `table_w`, `table_grad` and `table_loss`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`table_w` comes from your `batch_gd`. The other two apply `grad_w` and `train_loss` to each weight.

</details>

In [ ]:
# Exercise 4.6: running this cell checks your answer
table_w = ...  # your answer
table_grad = ...  # your answer
table_loss = ...  # your answer



<font size="4"><b>Your turn: the fourth update</b></font>

After the third update $w = 1.75$ and the gradient there is $-1.25$. Find the weight after one more update with step size 0.1.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Subtract the step size times the gradient. Mind the two minus signs.

</details>

In [ ]:
# Exercise 4.7: running this cell checks your answer
w_4 = ...  # your answer



<font size="4"><b>Half the gap each time</b></font>

Run 10 updates from 0 with step size 0.1. Make `gap_ratio`, the 10 ratios of the gap to 2 after each update divided by the gap before it.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The gaps are `2 - ws`. Divide the array from position 1 on by the array up to the last position.

</details>

In [ ]:
# Exercise 4.8: running this cell checks your answer
ws_10 = ...  # your answer
gap_ratio = ...  # your answer



<font size="4"><b>From the other side</b></font>

Start at $w = 4$, where the line is too steep. Give the gradient there and the weight after one update with step size 0.1.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Use your `grad_w`. Every residual is positive here.

</details>

In [ ]:
# Exercise 4.9: running this cell checks your answer
grad_4 = ...  # your answer
w_from4 = ...  # your answer



<font size="4"><b>When to stop</b></font>

Write `gd_until(w0, eta, tol, x, y)` that keeps updating while the size of the gradient is at least `tol`, and returns `(w, updates)`: the final weight and how many updates it took.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

A `while abs(grad_w(w, x, y)) >= tol:` loop with a counter.

</details>

In [ ]:
# Exercise 4.10: running this cell checks your answer
def gd_until(w0: float, eta: float, tol: float, x: np.ndarray, y: np.ndarray) -> tuple[float, int]:
    ...  # your code here



<font size="4"><b>The loss curve</b></font>

Make `curve_01`, the training loss at every weight of 20 updates from 0 with step size 0.1 (21 numbers). The loss curve appears when the check passes.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Your `batch_gd`, then your `train_loss` at each weight.

</details>

In [ ]:
# Exercise 4.11: running this cell checks your answer
curve_01 = ...  # your answer



## <font color="#d0309f">Choosing the step size</font>

Plot the slope of the bowl under the bowl itself. For the squared loss it is a straight line that crosses 0 at the bottom, $w = 2$. Its steepness is the **curvature**: how fast the slope changes when the weight moves. For this model it is twice the mean of $x^2$:
$$c = \frac{2}{n} \sum_i x_i^2 = 2 \times 2.5 = 5.$$

So the gradient is the curvature times the distance to the bottom, $c\,(w - w^\star)$. One update then gives
$$w_{\text{new}} - w^\star = (1 - \eta c)\,(w - w^\star).$$
Every update multiplies the distance to the bottom by the **multiplier** $1 - \eta c$.

* Between 0 and 1: the ball creeps down one side.
* Between -1 and 0: it overshoots, bounces from side to side, and settles.
* Exactly -1: it jumps between two mirror points forever.
* Below -1: every jump lands further out, and the loss explodes.

The run **converges** (settles on an answer) when the multiplier lies strictly between -1 and 1, and **diverges** (runs away) outside. The edge is $\eta = 2 / c$. For the five flats that is 0.4, the largest stable step.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharey=True)
for ax, (eta, w0, steps) in zip(axes, [(0.05, 0.0, 12), (0.3, 0.0, 6), (0.4, 1.0, 4), (0.5, 1.6, 5)]):
    draw_bowl(ax, _gd(w0, eta, steps), f"step size {eta}: multiplier {1 - eta * 5:g}")
plt.tight_layout()
plt.show()



<font size="4"><b>The slope line</b></font>

Compute the gradient at $w = 0, 1, 2, 3, 4$ as an array `slopes_04`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Your `grad_w` in a list comprehension.

</details>

In [ ]:
# Exercise 5.1: running this cell checks your answer
slopes_04 = ...  # your answer



<font size="4"><b>The curvature</b></font>

Write `curvature(x)` for the model $w x$: twice the mean of $x^2$.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

One line with `np.mean`.

</details>

In [ ]:
# Exercise 5.2: running this cell checks your answer
def curvature(x: np.ndarray) -> float:
    return ...  # your code here



<font size="4"><b>The largest stable step</b></font>

Compute `eta_max`, the largest stable step for the five flats, from your `curvature`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

2 divided by the curvature.

</details>

In [ ]:
# Exercise 5.3: running this cell checks your answer
eta_max = ...  # your answer



<font size="4"><b>The multiplier</b></font>

Write `multiplier(eta, c)` and use it to make `mults`, the multipliers of the step sizes in `ETAS` for curvature 5.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

$1 - \eta c$, for each step size in the list.

</details>

In [ ]:
# Exercise 5.4: running this cell checks your answer
ETAS = [0.05, 0.2, 0.3, 0.4, 0.5]


def multiplier(eta: float, c: float) -> float:
    return ...  # your code here


mults = ...  # your answer



<font size="4"><b>Which ones settle</b></font>

Make `settles`, a list of booleans, one per step size in `ETAS`: `True` when the run converges.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

The size of the multiplier must be strictly less than 1.

</details>

In [ ]:
# Exercise 5.5: running this cell checks your answer
settles = ...  # your answer



<font size="4"><b>Predict, then run</b></font>

From $w = 0$ with step size 0.3, predict the weight after 3 updates with the multiplier, without running gradient descent. Store it in `w3_predicted`. Then run your `batch_gd` and store its answer in `w3_run`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

The distance starts at $0 - 2 = -2$ and is multiplied by the multiplier 3 times. Add the result to 2.

</details>

In [ ]:
# Exercise 5.6: running this cell checks your answer
w3_predicted = ...  # your answer
w3_run = ...  # your answer



<font size="4"><b>Other data</b></font>

Two examples have inputs 1 and 3 (the targets don't matter for the curvature). Give their curvature and largest stable step as `(c_13, eta_13)`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

The mean of $x^2$ is $(1 + 9) / 2$.

</details>

In [ ]:
# Exercise 5.7: running this cell checks your answer
c_13 = ...  # your answer
eta_13 = ...  # your answer



<font size="4"><b>Your turn: step size 0.2</b></font>

Step size 0.2 has multiplier 0 on the flats. Run one update from $w = -3$ and one from $w = 7$, and store the two results as `landings`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Your `batch_gd` with 1 step, then take the last entry, for each start.

</details>

In [ ]:
# Exercise 5.8: running this cell checks your answer
landings = ...  # your answer



<font size="4"><b>Counting to an explosion</b></font>

From $w = 1.6$ with step size 0.5, the distance to the bottom grows by a factor of 1.5 each update. After how many updates does the training loss first go above 100? Use your `batch_gd` and `train_loss`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Run plenty of updates, then find the first index whose loss is above 100.

</details>

In [ ]:
# Exercise 5.9: running this cell checks your answer
ws_boom = ...  # your answer
updates_to_100 = ...  # your answer



<font size="4"><b>Two weights</b></font>

Add a bias, so the model is $w x + b$. The loss is now a bowl over two weights, and it has a curvature in each of two directions: the **eigenvalues** of the matrix `H2` below. Compute them with `np.linalg.eigvalsh`, and the largest stable step, which the sharpest direction sets.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`eigvalsh` returns the values in increasing order. Use the largest one in $2 / c$.

</details>

In [ ]:
# Exercise 5.10: running this cell checks your answer
H2 = 2 * np.array([[np.mean(X5 ** 2), np.mean(X5)], [np.mean(X5), 1.0]])
curvs_2 = ...  # your answer
eta_max_2 = ...  # your answer



<font size="4"><b>Find it by trial</b></font>

Find the largest stable step without the formula. For each step size in `TRIAL_ETAS`, run 50 updates from 0 and call it stable if the final loss is below the starting loss. Store the largest stable one in `eta_by_trial`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Loop over the step sizes, keep the ones that pass, and take the maximum.

</details>

In [ ]:
# Exercise 5.11: running this cell checks your answer
TRIAL_ETAS = np.round(np.arange(0.01, 0.61, 0.01), 2)
eta_by_trial = ...  # your answer



## <font color="#0077ff">Stochastic gradient descent</font>

Batch gradient descent reads every example before it moves once. With a million examples, every update needs a million residuals, and a run needs thousands of updates.

**Stochastic gradient descent**, or **SGD**, picks one example, computes the gradient of that example's loss alone, $2(w x_i - y_i)\,x_i$, and updates at once. Stochastic means chosen at random. The cost of an update no longer depends on the size of the data set.

Each flat has its own parabola, with its own bottom at $y_i / x_i$: the weight that fits that flat exactly. The training loss is the average of the five. So the single-example gradients average to the full gradient: an SGD step is right on average and off in any single draw.

One full pass through the training data is an **epoch**. In one epoch, batch gradient descent updates once and SGD updates once per example. In practice the examples are **shuffled** (put in a new random order) before each epoch.

In [ ]:
fig, ax = plt.subplots()
grid = np.linspace(0.5, 3.5, 200)
for i, (xi, yi) in enumerate(zip(X5, Y5)):
    ax.plot(grid, (grid * xi - yi) ** 2, lw=1.4, label=f"flat {i + 1}: bottom at {yi / xi:.2f}")
ax.plot(grid, [_loss(g) for g in grid], color=PALETTE["ink"], lw=4, label="training loss (the average)")
ax.set_ylim(0, 6); ax.set_xlabel("w"); ax.set_ylabel("loss")
ax.legend(fontsize=7)
ax.set_title("One parabola per flat, each with its own bottom")
plt.show()



<font size="4"><b>Each flat&#x27;s own bottom</b></font>

Compute `bottoms`, the weight that fits each flat exactly.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

The line $w x$ passes through $(x_i, y_i)$ when $w = y_i / x_i$.

</details>

In [ ]:
# Exercise 6.1: running this cell checks your answer
bottoms = ...  # your answer



<font size="4"><b>One example&#x27;s gradient</b></font>

Write `grad_one(w, x, y)`: the gradient of a single example's squared loss, for plain numbers.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

2 times the residual times the input.

</details>

In [ ]:
# Exercise 6.2: running this cell checks your answer
def grad_one(w: float, x: float, y: float) -> float:
    return ...  # your code here



<font size="4"><b>Scattered around the full gradient</b></font>

At $w = 1$, make `singles_1`, the 5 single-example gradients, and `mean_singles`, their mean.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Your `grad_one` over `zip(X5, Y5)`, then `np.mean`.

</details>

In [ ]:
# Exercise 6.3: running this cell checks your answer
singles_1 = ...  # your answer
mean_singles = ...  # your answer



<font size="4"><b>One SGD update</b></font>

From $w = 1$, take one SGD update on the fourth flat with step size 0.1.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

The fourth flat is at index 3.

</details>

In [ ]:
# Exercise 6.4: running this cell checks your answer
w_sgd4 = ...  # your answer



<font size="4"><b>An SGD pass</b></font>

Write `sgd_pass(w0, eta, order, x, y)`: one SGD update per index in `order`, in that order. Return the array of every weight visited, the start included.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

The same shape as your `batch_gd`, with `grad_one(ws[-1], x[i], y[i])` in place of the full gradient.

</details>

In [ ]:
# Exercise 6.5: running this cell checks your answer
def sgd_pass(w0: float, eta: float, order, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    ...  # your code here



<font size="4"><b>One epoch each</b></font>

From $w = 0$ with step size 0.1, run one SGD pass through the flats in order, and one batch update. Store the two final weights as `sgd_end` and `batch_end`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`range(5)` is the order. One batch update is your `batch_gd` with 1 step.

</details>

In [ ]:
# Exercise 6.6: running this cell checks your answer
sgd_end = ...  # your answer
batch_end = ...  # your answer



<font size="4"><b>Your turn: from the bottom</b></font>

From $w = 2$, take one SGD update on the second flat (area 1, price 2.3) with step size 0.1.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

The residual is $2 - 2.3$.

</details>

In [ ]:
# Exercise 6.7: running this cell checks your answer
w_from2 = ...  # your answer



<font size="4"><b>Who pulls down at the bottom</b></font>

At $w = 2$ the full gradient is 0. Give the list of indices of the flats whose own gradient is positive there, so an SGD update on them pulls the weight down.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Compute all five single gradients at 2 and keep the indices where the value is above 0.

</details>

In [ ]:
# Exercise 6.8: running this cell checks your answer
pull_down = ...  # your answer



<font size="4"><b>Shuffled epochs</b></font>

Write `sgd_shuffled(w0, eta, epochs, seed, x, y)`. Make `r = np.random.default_rng(seed)`. For each epoch, draw `order = r.permutation(len(x))`, a random order of the indices, and run one pass in that order. Return every weight visited as one array. The path appears when the check passes.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Keep one list of weights across all epochs. Your `sgd_pass` returns the start too, so drop its first entry when you join passes.

</details>

In [ ]:
# Exercise 6.9: running this cell checks your answer
def sgd_shuffled(w0: float, eta: float, epochs: int, seed: int, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    ...  # your code here



<font size="4"><b>A shrinking step size</b></font>

The usual cure is to lower the step size as training goes on. Run 60 epochs from 0 with seed 3 in two ways: with a fixed step size of 0.1 (your `sgd_shuffled`), and with step size $0.1 / (1 + t/20)$ at update $t = 0, 1, 2, \dots$, using the same permutations. Store the standard deviation of the last 50 weights of each run as `spread_fixed` and `spread_decay`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

For the shrinking run, write the loop yourself: `r = np.random.default_rng(3)`, a counter `t` that goes up by 1 after every update, and `np.std(ws[-50:])` at the end.

</details>

In [ ]:
# Exercise 6.10: running this cell checks your answer
spread_fixed = ...  # your answer
spread_decay = ...  # your answer



<font size="4"><b>The cost per epoch</b></font>

A data set has a million examples. Give `(read_batch, read_sgd, updates_batch, updates_sgd)`: the examples read per update by batch gradient descent and by SGD, then the updates per epoch of each.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Batch reads everything for one update. SGD reads one example per update.

</details>

In [ ]:
# Exercise 6.11: running this cell checks your answer
cost_million = ...  # your answer



## <font color="#7b2ff7">Mini-batches</font>

A **mini-batch** is a small group of examples drawn from the training data. The update uses the average of their gradients. The number of examples in the group is the **batch size**.

Batch size is a setting between two extremes. A batch of 1 is plain SGD: the cheapest and noisiest steps. A batch of the whole data set is batch gradient descent: exact steps, at full cost.

Averaging cancels part of the noise. The cost of an update grows in step with the batch size, and the noise (the standard deviation of the mini-batch gradient) shrinks with the square root: a batch 100 times larger is about 10 times less noisy. A computer can also process 32 or 64 examples in one array operation in barely more time than one, so batch sizes from about 30 to a few hundred are common. In everyday use, the name SGD covers the mini-batch version too.

In [ ]:
singles = _grad1(1.0, _X5, _Y5)
pairs = [singles[list(p)].mean() for p in itertools.combinations(range(5), 2)]
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.scatter(singles, np.ones(5), s=80, color=PALETTE["pink"], label="single examples")
ax.scatter(pairs, np.zeros(10), s=80, color=PALETTE["cyan"], label="averages of two")
ax.axvline(_grad(1.0), color=PALETTE["amber"], ls="--", label="full gradient, -5")
ax.set_yticks([0, 1], ["pairs", "singles"]); ax.set_ylim(-0.7, 1.7)
ax.set_xlabel("gradient at w = 1"); ax.legend(fontsize=8, loc="lower left")
ax.set_title("Averaging two examples pulls the gradient toward the full one")
plt.show()



<font size="4"><b>A pair of flats</b></font>

From $w = 1$, average the gradients of the first and fourth flats, and take one update with step size 0.1.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Indices 0 and 3. Average the two values of your `grad_one`.

</details>

In [ ]:
# Exercise 7.1: running this cell checks your answer
g_pair = ...  # your answer
w_pair = ...  # your answer



<font size="4"><b>The mini-batch gradient</b></font>

Write `minibatch_grad(w, x, y, idx)`: the mean of the single-example gradients over the indices in `idx`. Use array indexing, no loop.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`x[idx]` and `y[idx]` pick the rows. Then the same formula as `grad_w`.

</details>

In [ ]:
# Exercise 7.2: running this cell checks your answer
def minibatch_grad(w: float, x: np.ndarray, y: np.ndarray, idx) -> float:
    idx = np.asarray(idx)
    return ...  # your code here



<font size="4"><b>Every pair</b></font>

At $w = 1$, compute the mini-batch gradient of every pair of flats, sorted from smallest to largest, as an array `pair_grads`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`itertools.combinations(range(5), 2)` lists the 10 pairs.

</details>

In [ ]:
# Exercise 7.3: running this cell checks your answer
pair_grads = ...  # your answer



<font size="4"><b>How much noise is left</b></font>

Compute the standard deviation of the 5 single gradients and of the 10 pair gradients at $w = 1$ (`np.std` of each), and their ratio pairs over singles.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

You have `singles_1` from section 6 and `pair_grads` from the last exercise.

</details>

In [ ]:
# Exercise 7.4: running this cell checks your answer
noise_ratio = ...  # your answer



<font size="4"><b>Two flats that cancel</b></font>

At $w = 2$, compute the mini-batch gradient of the two flats with the same area (indices 1 and 2).

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Your `minibatch_grad` with `[1, 2]`.

</details>

In [ ]:
# Exercise 7.5: running this cell checks your answer
g_cancel = ...  # your answer



<font size="4"><b>Updates per epoch</b></font>

Give the updates in one epoch for 1,200 examples with batch size 40 (`updates_a`), and for 1,000 examples with batch size 64 (`updates_b`), where the last batch holds whatever is left. Also give the size of that last batch, `last_b`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Divide, and round up when it doesn't divide evenly. `math.ceil` rounds up.

</details>

In [ ]:
# Exercise 7.6: running this cell checks your answer
updates_a = ...  # your answer
updates_b = ...  # your answer
last_b = ...  # your answer



<font size="4"><b>The square-root rule</b></font>

By the square-root rule, how many times less noisy is a batch of 400 than a batch of 4? And what batch size halves the noise of a batch of 8? Give `(times_less, batch_half)`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Noise goes with 1 over the square root of the batch size. Halving it needs a batch 4 times larger.

</details>

In [ ]:
# Exercise 7.7: running this cell checks your answer
times_less, batch_half = ..., ...  # your answer



<font size="4"><b>Measure the rule</b></font>

`X_BIG` and `Y_BIG` hold 20,000 examples. For each batch size in `BS`, draw 2,000 mini-batches of random indices (with `rng_b.integers(0, 20000, size=B)`), compute each one's gradient at $w = 1$ with your `minibatch_grad`, and store the standard deviation of those gradients in `noise_by_B`. The plot appears when the check passes.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

An outer loop over `BS` and an inner loop of 2,000 draws. Collect the gradients in a list, then `np.std`.

</details>

In [ ]:
# Exercise 7.8: running this cell checks your answer
X_BIG = np.random.default_rng(10).uniform(0.3, 3.0, 20000)
Y_BIG = 2.0 * X_BIG + np.random.default_rng(11).normal(0, 0.3, 20000)
BS = [1, 4, 16, 64]
rng_b = np.random.default_rng(12)
noise_by_B = ...  # your answer



<font size="4"><b>Cutting an epoch into batches</b></font>

Write `make_batches(n, B, r)`: draw one permutation of `range(n)` with `r.permutation(n)` and cut it into consecutive pieces of size `B`, the last piece holding whatever is left. Return the list of index arrays.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Slice the permutation at `0, B, 2B, ...` with `range(0, n, B)`.

</details>

In [ ]:
# Exercise 7.9: running this cell checks your answer
def make_batches(n: int, B: int, r: np.random.Generator) -> list[np.ndarray]:
    ...  # your code here



<font size="4"><b>Batches in PyTorch</b></font>

PyTorch's `DataLoader` cuts a data set into mini-batches. Wrap the flats in `torch.utils.data.TensorDataset` and make a `DataLoader` with `batch_size=2` and `shuffle=False`. Store the list of batch sizes it produces in `loader_sizes`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Loop `for xb, yb in loader:` and record `len(xb)`.

</details>

In [ ]:
# Exercise 7.10: running this cell checks your answer
loader_sizes = ...  # your answer



## <font color="#e52e71">Convexity</font>

Our loss has been a bowl with a single lowest point, so downhill from anywhere leads to the same bottom. Not every loss has that shape. The curve below has two dips. A point lower than everything near it is a **local minimum**; the lowest point of all is the **global minimum**. Gradient descent only feels the slope where it stands, so it cannot tell the two apart, and the result depends on the start.

A function shaped like a single bowl is **convex**. The test uses a **chord**: pick two points on the curve and join them with a straight line. If no chord ever passes below the curve, the function is convex. In symbols, for every $a$, $b$ and every $t$ between 0 and 1,
$$f\big(t a + (1 - t) b\big) \le t f(a) + (1 - t) f(b).$$

For a convex function every local minimum is a global minimum. The squared loss of a linear predictor is convex: each example's loss is a parabola opening upward, and an average of those opens upward too. So for linear regression, with a stable step size, gradient descent reaches the best weights from any start. Neural networks give up this guarantee.

In [ ]:
def descend(f, w0, eta=0.02, steps=400, h=1e-5):
    """Gradient descent on any function of one number, with the slope measured by a central difference."""
    w = w0
    path = [w]
    for _ in range(steps):
        w -= eta * (f(w + h) - f(w - h)) / (2 * h)
        path.append(w)
    return np.array(path)


grid = np.linspace(-2.3, 2.3, 300)
fig, ax = plt.subplots()
ax.plot(grid, _two_dip(grid), color=PALETTE["violet"])
for w0, colour in [(-2.3, PALETTE["mint"]), (2.3, PALETTE["pink"])]:
    p = descend(_two_dip, w0)[::8]
    ax.plot(p, _two_dip(p), "o", color=colour, ms=4, label=f"start at {w0}")
a, b = -1.48, 1.33
ax.plot([a, b], [_two_dip(a), _two_dip(b)], color=PALETTE["amber"], ls="--", label="a chord that passes below")
ax.set_xlabel("w"); ax.set_ylabel("loss"); ax.legend(fontsize=8)
ax.set_title("Two dips: where you stop depends on where you start")
plt.show()



<font size="4"><b>Two starts, two answers</b></font>

Run `descend(_two_dip, ...)` from $-2.3$ and from $2.3$, and store the final positions as `end_left` and `end_right`. Then say which start finds the global minimum: `"left"` or `"right"`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`descend` returns the whole path; the last entry is where it stopped. Compare the heights `_two_dip(...)` of the two ends.

</details>

In [ ]:
# Exercise 8.1: running this cell checks your answer
end_left = ...  # your answer
end_right = ...  # your answer
global_from = ...  # your answer



<font size="4"><b>The chord gap</b></font>

Write `chord_gap(f, a, b, t)`: the height of the chord minus the height of the curve at the point $t a + (1 - t) b$. A negative gap means the chord passes below the curve.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

The chord's height is $t f(a) + (1 - t) f(b)$.

</details>

In [ ]:
# Exercise 8.2: running this cell checks your answer
def chord_gap(f, a: float, b: float, t: float) -> float:
    return ...  # your code here



<font size="4"><b>The failing chord</b></font>

Compute the chord gap of `_two_dip` between your two end points, at $t = 0.5$.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Your `chord_gap` with `end_left` and `end_right`.

</details>

In [ ]:
# Exercise 8.3: running this cell checks your answer
gap_dips = ...  # your answer



<font size="4"><b>A convexity test</b></font>

Write `is_convex(f, lo, hi, n=81)`: take `n` evenly spaced points from `lo` to `hi`, and return `False` if any pair of them, at any $t$ in `(0.25, 0.5, 0.75)`, has a chord gap below `-1e-9`. Otherwise return `True`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`np.linspace(lo, hi, n)` and `itertools.combinations(xs, 2)` for the pairs. The small `-1e-9` allows for rounding.

</details>

In [ ]:
# Exercise 8.4: running this cell checks your answer
def is_convex(f, lo: float, hi: float, n: int = 81) -> bool:
    ...  # your code here



<font size="4"><b>Your turn: three curves</b></font>

Test three curves: $e^x$ on $[-2, 2]$, the hill $-x^2$ on $[-2, 2]$, and the wave $\sin x$ on $[-4, 4]$. Store the three results as a tuple.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`np.exp`, `lambda x: -x * x` and `np.sin` with your `is_convex`.

</details>

In [ ]:
# Exercise 8.5: running this cell checks your answer
three_curves = ...  # your answer



<font size="4"><b>Four more curves</b></font>

Test, each on $[-2, 2]$: $\cosh x$ (`np.cosh`), $x^3$, $\max(0, x)$ and $\cos x$. Store the four results as a tuple.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Predict first: which of these could hold water? Then run your test.

</details>

In [ ]:
# Exercise 8.6: running this cell checks your answer
four_curves = ...  # your answer



<font size="4"><b>The flats&#x27; loss is convex</b></font>

Test the training loss of the flats as a function of $w$ on $[-2, 6]$ with your `is_convex` and `train_loss`. Store the result in `flats_convex`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Wrap it: `lambda w: train_loss(w, X5, Y5)`.

</details>

In [ ]:
# Exercise 8.7: running this cell checks your answer
flats_convex = ...  # your answer



<font size="4"><b>How a curve bends</b></font>

The **second derivative** measures how the slope changes. Write `second_diff(f, x, h=1e-4)` with the central formula $\big(f(x + h) - 2 f(x) + f(x - h)\big) / h^2$. Then make `bend_flats`, the second difference of the flats' training loss at $w = -3, 0, 2, 10$, and `bend_dip`, the second difference of `_two_dip` at 0.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

For the flats, wrap `train_loss` in a lambda as before and loop over the four points.

</details>

In [ ]:
# Exercise 8.8: running this cell checks your answer
def second_diff(f, x: float, h: float = 1e-4) -> float:
    return ...  # your code here


bend_flats = ...  # your answer
bend_dip = ...  # your answer



<font size="4"><b>Count the dips</b></font>

On the grid `DIP_GRID`, count the interior points where `_two_dip` is lower than both neighbours. Store the count in `n_dips`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

With `v = _two_dip(DIP_GRID)`, compare `v[1:-1]` with `v[:-2]` and with `v[2:]`.

</details>

In [ ]:
# Exercise 8.9: running this cell checks your answer
DIP_GRID = np.linspace(-2.5, 2.5, 1001)
v_dip = _two_dip(DIP_GRID)
n_dips = ...  # your answer



## <font color="#00897b">Reading loss curves</font>

A loss curve plots the training loss after every update. It is the first thing to look at when a run misbehaves. Four shapes cover most cases:

* **Healthy**: a steep drop, a bend, and a flat tail.
* **Diverging**: the loss grows with every update, faster and faster. The step size is above the stable threshold; the cure is a smaller step size.
* **Too slow**: a shallow, almost straight fall. The step size is too small; the cure is to raise it.
* **Noisy**: a drop, then jitter up and down. That is SGD with a step size too large for its noise. A smaller step size, or a larger batch, calms it.

The shape is half of the reading. The other half is the scale on the axis: a stretched axis can make a slow run look steep. Also check where the curve flattens. A high floor means the family or the features are too weak, and no step size will fix that.

Divergence shows within the first few updates, so a short trial of 20 or 50 updates is enough to rule out a step size.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 5.6))
colours = [PALETTE["mint"], PALETTE["pink"], PALETTE["amber"], PALETTE["cyan"]]
for ax, (name, c), colour in zip(axes.flat, _TEACH_CURVES.items(), colours):
    ax.plot(c, "-o", color=colour, ms=3)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("update")
plt.tight_layout()
plt.show()



<font size="4"><b>A loss curve function</b></font>

Write `loss_curve(w0, eta, steps, x, y)`: the list of training losses at every weight of a batch gradient descent run, the start included. Use your `batch_gd` and `train_loss`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

A list comprehension over the weights from `batch_gd`.

</details>

In [ ]:
# Exercise 9.1: running this cell checks your answer
def loss_curve(w0: float, eta: float, steps: int, x: np.ndarray, y: np.ndarray) -> list[float]:
    return ...  # your code here



<font size="4"><b>How fast it diverges</b></font>

For the run with step size 0.45, divide the last loss by the one before it. Compare it with the square of the multiplier, $(1 - 0.45 \times 5)^2$. Store both as `(ratio_div, mult_sq)`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The loss above its floor grows with the square of the distance, so each update multiplies it by about the square of the multiplier.

</details>

In [ ]:
# Exercise 9.2: running this cell checks your answer
c_div = ...  # your answer
ratio_div, mult_sq = ..., ...  # your answer



<font size="4"><b>How slow is slow</b></font>

For the run with step size 0.002 from 0, what percentage of the starting loss is gone after 20 updates?

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

$100 \times (1 - \text{last} / \text{first})$.

</details>

In [ ]:
# Exercise 9.3: running this cell checks your answer
c_slow = ...  # your answer
drop_slow = ...  # your answer



<font size="4"><b>Your turn: name the curves</b></font>

The cell below draws four curves, A to D, from four runs on the flats. Name each one with `"healthy"`, `"diverging"`, `"slow"` or `"noisy"`. Look at the shape, then at the scale.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

One of them looks steep until you read its axis.

</details>

In [ ]:
# Exercise 9.4: running this cell checks your answer
fig, axes = plt.subplots(1, 4, figsize=(12, 2.8))
for ax, (name, c) in zip(axes, _TRY_CURVES.items()):
    ax.plot(c, "-o", color=PALETTE["violet"], ms=3)
    ax.set_title(f"curve {name}", fontsize=10)
plt.tight_layout()
plt.show()

curve_names = ...  # your answer



<font size="4"><b>A diagnosis by rule</b></font>

Write `diagnose(losses)` with these rules, in this order: if the last loss is above the first, `"diverging"`; if the loss goes up between consecutive updates (by more than `1e-9`) at least 3 times, `"noisy"`; if the last loss is more than half of the first, `"slow"`; otherwise `"healthy"`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`np.diff` gives the changes between consecutive losses. Count how many are above `1e-9`.

</details>

In [ ]:
# Exercise 9.5: running this cell checks your answer
def diagnose(losses) -> str:
    ...  # your code here



<font size="4"><b>The cure</b></font>

The diverging run used step size 0.45. Rerun it with 0.1 and store the last loss as `fixed_last`, and your diagnosis of the new curve as `fixed_label`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Your `loss_curve`, then your `diagnose`.

</details>

In [ ]:
# Exercise 9.6: running this cell checks your answer
c_fixed = ...  # your answer
fixed_last, fixed_label = ..., ...  # your answer



<font size="4"><b>The scale trap</b></font>

A run with step size 0.001 for 10 updates from 0 looks like a fine steep drop on a stretched axis. Compute the percentage of the loss it removed, and your diagnosis.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The same percentage formula as before, on a 10-update curve.

</details>

In [ ]:
# Exercise 9.7: running this cell checks your answer
c_scale = ...  # your answer
drop_scale, label_scale = ..., ...  # your answer



<font size="4"><b>A high floor</b></font>

A model with only a bias, price $= b$, has its lowest loss at $b = \bar{y}$, the mean price. Compute that floor, `floor_bias`, and compare it with the floor of the model $w x$, `floor_w`, which is the loss at $w = 2$.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The mean of the squared gaps between each price and the mean price.

</details>

In [ ]:
# Exercise 9.8: running this cell checks your answer
floor_bias = ...  # your answer
floor_w = ...  # your answer



<font size="4"><b>A short trial</b></font>

For each step size in `SHORT_ETAS`, run 20 updates from 0 and rule it out if the last loss is above the first. Store the list of ruled-out step sizes in `ruled_out`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

A list comprehension with your `loss_curve`.

</details>

In [ ]:
# Exercise 9.9: running this cell checks your answer
SHORT_ETAS = [0.05, 0.1, 0.3, 0.39, 0.41, 0.6]
ruled_out = ...  # your answer



## <font color="#ff4d6d">The loop in NumPy and PyTorch</font>

NumPy fits all of batch gradient descent in a few lines. The data form a matrix $X$ with one row per example and one column per feature, of **shape** $(n, d)$. The targets $y$ have shape $(n,)$ and the weights $w$ have shape $(d,)$.

One update takes four array operations:

* the predictions $Xw$: shape $(n, d)$ times $(d,)$ gives $(n,)$, one per example;
* the **residual vector** $r = Xw - y$, shape $(n,)$;
* the gradient $\frac{2}{n} X^\top r$: the **transpose** $X^\top$ has shape $(d, n)$, so the result has shape $(d,)$, one slope per weight;
* the update $w - \eta \times$ gradient, shape $(d,)$.

Put them inside a loop, starting from $w = 0$, and that is batch gradient descent for any number of features. The loss is the mean of the squared residuals, a single number.

None of these steps loops over the examples in Python. Each is one array operation over all of them. Code written this way is **vectorized**, and it is much faster than a Python loop.

The shapes also catch the most common bug. If $y$ is loaded as a column of shape $(n, 1)$, then $Xw - y$ silently **broadcasts**: NumPy stretches both arrays to $(n, n)$, and the loss is wrong without any error. One printed shape catches it.

In [ ]:
def grad_python_loop(X, y, w):
    """The gradient with plain Python loops, one example and one feature at a time."""
    n, d = len(X), len(X[0])
    g = [0.0] * d
    for i in range(n):
        pred = sum(X[i][j] * w[j] for j in range(d))
        for j in range(d):
            g[j] += 2 * (pred - y[i]) * X[i][j] / n
    return g


X_T = np.random.default_rng(0).normal(size=(20000, 10))
y_T = X_T @ np.linspace(-1, 1, 10)
w_T = np.zeros(10)
t0 = time.perf_counter(); grad_python_loop(X_T.tolist(), y_T.tolist(), w_T.tolist()); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); (2 / 20000) * X_T.T @ (X_T @ w_T - y_T); t_vec = time.perf_counter() - t0
print(f"Python loop: {1000 * t_loop:.1f} ms   vectorized: {1000 * t_vec:.2f} ms   about {t_loop / t_vec:.0f} times faster")

fig, ax = plt.subplots(figsize=(6, 2.4))
ax.barh(["vectorized", "Python loop"], [1000 * t_vec, 1000 * t_loop], color=[PALETTE["mint"], PALETTE["pink"]])
ax.set_xlabel("milliseconds for one gradient, 20,000 examples")
ax.set_title("The same gradient, two ways")
plt.show()



<font size="4"><b>The flats as a matrix</b></font>

Make `X_col`, the five areas as a matrix of shape `(5, 1)`, and `pred_shape`, the shape of `X_col @ np.zeros(1)`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`reshape(-1, 1)` turns a vector into a column. `.shape` reads a shape.

</details>

In [ ]:
# Exercise 10.1: running this cell checks your answer
X_col = ...  # your answer
pred_shape = ...  # your answer



<font size="4"><b>The loop as a function</b></font>

Write `fit(X, y, eta, steps)` with the lines from the text. Return `(w, losses)`, where `losses` holds the training loss computed inside the loop before each update (`steps` numbers).

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Read `n, d` from `X.shape`. The loss is the mean of the squared residual vector.

</details>

In [ ]:
# Exercise 10.2: running this cell checks your answer
def fit(X: np.ndarray, y: np.ndarray, eta: float, steps: int) -> tuple[np.ndarray, list[float]]:
    ...  # your code here



<font size="4"><b>Run it on the flats</b></font>

Run your `fit` on `X_col` and `Y5` for 20 updates with step size 0.1. Store the weight array in `w_flats`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Take the first item of what `fit` returns.

</details>

In [ ]:
# Exercise 10.3: running this cell checks your answer
w_flats = ...  # your answer



<font size="4"><b>Your turn: shapes</b></font>

With 100 examples and 3 features, what are the shapes of the residual vector and of the gradient? Give a tuple of two shape tuples.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

One residual per example, one slope per weight.

</details>

In [ ]:
# Exercise 10.4: running this cell checks your answer
shapes_100 = ...  # your answer



<font size="4"><b>The broadcasting bug</b></font>

Suppose the prices were loaded as a column, `Y_col = Y5.reshape(5, 1)`. With `w = np.array([2.0])`, give the shape of `X_col @ w - Y_col`, the loss computed from it with `np.mean(... ** 2)`, and the right loss.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Compute it and look at the shape. The right loss uses `Y5`.

</details>

In [ ]:
# Exercise 10.5: running this cell checks your answer
Y_col = Y5.reshape(5, 1)
bug_shape = ...  # your answer
bug_loss = ...  # your answer
right_loss = ...  # your answer



<font size="4"><b>A guard</b></font>

Write `residuals(X, w, y)` that returns `X @ w - y` with shape `(n,)` even when `y` arrives as a column of shape `(n, 1)`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`y.reshape(-1)` flattens a column. It leaves a vector unchanged.

</details>

In [ ]:
# Exercise 10.6: running this cell checks your answer
def residuals(X: np.ndarray, w: np.ndarray, y: np.ndarray) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>Add a bias</b></font>

Make `X_b`, the areas with a column of ones next to them, shape `(5, 2)`. Fit $w x + b$ with your `fit` for 3,000 updates of size 0.1, and store the weights in `wb_fit`. Compare with `np.linalg.lstsq`, which solves for the best weights directly, in `wb_best`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`np.column_stack([X5, np.ones(5)])`. `np.linalg.lstsq(X_b, Y5, rcond=None)` returns a tuple whose first item is the weights.

</details>

In [ ]:
# Exercise 10.7: running this cell checks your answer
X_b = ...  # your answer
wb_fit = ...  # your answer
wb_best = ...  # your answer



<font size="4"><b>Mini-batch SGD in NumPy</b></font>

Write `fit_sgd(X, y, eta, B, epochs, seed)`. Make `r = np.random.default_rng(seed)` and start from zero weights. Each epoch, draw `perm = r.permutation(n)`, cut it into consecutive batches of `B`, and update on each batch with the batch's own gradient, `2 / len(idx)` times $X_b^\top r_b$. Return the weights.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Two added lines compared with `fit`: pick the batch's indices, then take `X[idx]` and `y[idx]`. The other lines stay, with the batch size in place of `n`.

</details>

In [ ]:
# Exercise 10.8: running this cell checks your answer
def fit_sgd(X: np.ndarray, y: np.ndarray, eta: float, B: int, epochs: int, seed: int) -> np.ndarray:
    ...  # your code here



<font size="4"><b>Check the vectorized gradient</b></font>

Compare one vectorized gradient line with `grad_python_loop` on `X_b`, `Y5` at $w = (1, 0.5)$. Store the largest absolute difference in `max_gap`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`grad_python_loop` takes lists, so pass `.tolist()` versions, and wrap its result in `np.array`.

</details>

In [ ]:
# Exercise 10.9: running this cell checks your answer
w_chk = np.array([1.0, 0.5])
max_gap = ...  # your answer



<font size="4"><b>The gradient with einops</b></font>

`einops.einsum` names the axes. Compute $X^\top r$ for `X_b` at `w_chk` as `einops.einsum(X_b, r, "n d, n -> d")`, then scale it into the gradient.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The residual first, then the einsum, then multiply by `2 / 5`.

</details>

In [ ]:
# Exercise 10.10: running this cell checks your answer
grad_einops = ...  # your answer



<font size="4"><b>The same fit in PyTorch</b></font>

Fit $w x + b$ on the flats with PyTorch: a 64-bit weight tensor of zeros with `requires_grad=True`, `torch.optim.SGD` with step size 0.1, and 3,000 steps of zero, forward, backward, step. Store the weights as a NumPy array in `wb_torch`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`torch.zeros(2, dtype=torch.float64, requires_grad=True)`. The loss is `((Xt @ w - yt) ** 2).mean()`. End with `w.detach().numpy()`.

</details>

In [ ]:
# Exercise 10.11: running this cell checks your answer
wb_torch = ...  # your answer



## <font color="#1e88e5">The payoff</font>

Back to the two runs from the start: the same data, model and method, with step sizes 0.5 and 0.05. Now we can say what happens before running anything.

Square the five areas and average them: 2.5. Double that for the curvature: 5. Two divided by the curvature gives the largest stable step, 0.4. The step size 0.05 sits far below it, with multiplier 0.75, so the gap to the best weight shrinks at every update. The step size 0.5 sits above it, with multiplier -1.5, so the gap flips sign and grows.

The threshold belongs to the data, so it moves when the data change. One large input raises the curvature and lowers the threshold. That is why inputs are usually **rescaled** before training, so that no feature takes very large values, and why a loss curve that turns upward should make you think of the step size first.

In [ ]:
etas_axis = np.linspace(0, 0.6, 2)
fig, ax = plt.subplots(figsize=(7, 2.2))
ax.axvspan(0, 0.4, color=PALETTE["mint"], alpha=0.15)
ax.axvspan(0.4, 0.6, color=PALETTE["pink"], alpha=0.15)
ax.axvline(0.4, color=PALETTE["amber"], lw=2)
ax.scatter([0.05, 0.5], [0, 0], s=120, color=[PALETTE["mint"], PALETTE["pink"]], zorder=3)
ax.annotate("0.05: settles", (0.05, 0), (0.06, 0.4), color=PALETTE["mint"])
ax.annotate("0.5: explodes", (0.5, 0), (0.44, 0.4), color=PALETTE["pink"])
ax.annotate("2 / curvature = 0.4", (0.4, 0), (0.3, -0.6), color=PALETTE["amber"])
ax.set_yticks([]); ax.set_ylim(-1, 1); ax.set_xlim(0, 0.6)
ax.set_xlabel("step size")
ax.set_title("The five flats: stable steps in green")
plt.show()



<font size="4"><b>Predict from the inputs</b></font>

Using only `X5`, compute the curvature and the largest stable step, as `(c_flats, eta_flats)`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Your `curvature`, then 2 over it.

</details>

In [ ]:
# Exercise 11.1: running this cell checks your answer
c_flats = ...  # your answer
eta_flats = ...  # your answer



<font size="4"><b>The two verdicts</b></font>

Give the multipliers of 0.05 and 0.5 on the flats, and whether each run converges, as `(m_small, m_big, ok_small, ok_big)`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Your `multiplier` with curvature 5, then compare the sizes with 1.

</details>

In [ ]:
# Exercise 11.2: running this cell checks your answer
m_small, m_big = ..., ...  # your answer
ok_small, ok_big = ..., ...  # your answer



<font size="4"><b>Confirm by running</b></font>

Run 8 updates from 0 with each step size and store the two loss curves as `curve_small` and `curve_big`. The plot appears when the check passes.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Your `loss_curve`, twice.

</details>

In [ ]:
# Exercise 11.3: running this cell checks your answer
curve_small = ...  # your answer
curve_big = ...  # your answer



<font size="4"><b>One large flat</b></font>

Add a flat of 500 square metres that sold for 1.02 million: `X6` and `Y6` below. Compute the new curvature and largest stable step.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

The same two lines on `X6`.

</details>

In [ ]:
# Exercise 11.4: running this cell checks your answer
X6 = np.append(X5, 5.0)
Y6 = np.append(Y5, 10.2)
c6 = ...  # your answer
eta6 = ...  # your answer



<font size="4"><b>The best step turns bad</b></font>

Step size 0.2 was the best choice on five flats. Give its multiplier on the six flats, and the loss after 10 updates from 0 divided by the starting loss, as `(m6, growth6)`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Your `multiplier` with `c6`, and your `loss_curve` on `X6`, `Y6`.

</details>

In [ ]:
# Exercise 11.5: running this cell checks your answer
m6 = ...  # your answer
c_six = ...  # your answer
growth6 = ...  # your answer



<font size="4"><b>Rescale the inputs</b></font>

Divide the six areas by the largest one, so they lie between 0 and 1. Compute the curvature of the scaled inputs and its largest stable step, as `(c_scaled, eta_scaled)`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`X6 / X6.max()`, then your `curvature`.

</details>

In [ ]:
# Exercise 11.6: running this cell checks your answer
X6s = ...  # your answer
c_scaled = ...  # your answer
eta_scaled = ...  # your answer



<font size="4"><b>Back to the original units</b></font>

On the scaled inputs, step size $1 / c$ has multiplier 0, so one update from 0 lands on the best weight. Take that update into `w_scaled`. The model is then $w_{\text{scaled}} \cdot (x / 5)$, so the weight per original unit of area is `w_scaled / 5`. Store it in `w_orig` and check it against the formula from section 3 on the six flats.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

One update: `0 - (1 / c_scaled) * grad_w(0, X6s, Y6)`.

</details>

In [ ]:
# Exercise 11.7: running this cell checks your answer
w_scaled = ...  # your answer
w_orig = ...  # your answer
w_formula = ...  # your answer



<font size="4"><b>The threshold as a function</b></font>

Write `threshold_with(A)`: the largest stable step for the five flats plus one extra flat of area `A` (in hundreds of square metres). Then make `thresholds`, its value at every area in `AREAS`. The plot appears when the check passes.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`np.append(X5, A)`, then 2 over your `curvature`.

</details>

In [ ]:
# Exercise 11.8: running this cell checks your answer
AREAS = np.linspace(0, 8, 81)


def threshold_with(A: float) -> float:
    return ...  # your code here


thresholds = ...  # your answer



<font size="4"><b>How large before 0.2 breaks</b></font>

Find the largest area $A$ of an extra flat for which step size 0.2 is still stable on the six flats. Solve $2 / c \ge 0.2$ by hand, or search finely with your `threshold_with`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

The curvature is $2 (12.5 + A^2) / 6$, using the sum of the five squared areas, 12.5. Set $2 / c = 0.2$ and solve for $A$.

</details>

In [ ]:
# Exercise 11.9: running this cell checks your answer
A_limit = ...  # your answer



## <font color="#d84315">What you built</font>

* Every learner is four decisions: the family of predictors, the features, the loss and the optimizer. The first two are modeling, the last two are learning.
* A feature extractor turns an input into numbers. Categories become one-hot boxes, a bias feature adds a base value, and sparse vectors store only their non-zero entries.
* The squared loss averages the squared residuals. For a linear predictor it is a convex bowl, so every local minimum is the global one.
* Batch gradient descent updates with the full gradient. Each update multiplies the distance to the bottom by $1 - \eta c$, so the largest stable step is $2 / c$, with curvature $c = 2\,\overline{x^2}$.
* SGD updates on one example at a time: cheap and noisy. Mini-batches average a few examples, and the noise falls with the square root of the batch size.
* A loss curve tells healthy, diverging, too slow and noisy runs apart. Read the shape, then the scale, then the floor.
* In NumPy the whole loop is a few vectorized lines, and printing shapes catches broadcasting bugs.

The next lesson keeps the optimizer and changes the task: from predicting a number to choosing a label, with new losses and richer features.



Run the cell below any time to see how far you are.

In [ ]:
progress()

<font size="2" color="#8a84b3">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</font>